# NB19 — Held-Out Transformer Interpretation (Herbal Supplements)

**Purpose.** Descriptive predictive-dependence interpretation of the Transformer family: held-out semantic-group masking diagnostics (explicitly NOT causal contributions), epoch-history and learning-curve extraction from the fold checkpoints, and per-case diagnostics. Canonical performance is read ONLY from NB14, and the checkpoint-selection contract is asserted to be `validation_ndcg_at_5`. RankP metadata reconstruction writes only to the NB19 analysis directory; expected artifact unavailability activates the explicit Path B fallback, while unexpected programming or schema errors are re-raised — SELF-CHECK SC-2 validates this Path A/B exception-safety contract with synthetic tests. The Batch C2 guard asserts exact-item familiarity features are absent.
**Inputs.** NB13 run manifests, fold checkpoints, and interpretation manifests; NB14 canonical parquet + `pipeline_manifest.json`; NB15 `stage_allocation_manifest.json`.
**Outputs.** `transformer_masking_by_fold.csv`, `transformer_masking_summary.csv`, `transformer_semantic_group_stability.csv`, `transformer_interpretation_qc.csv`, `transformer_interpretation_scope_note.csv`, `reconstructed_s2p_feature_interpretation_manifest.json`, `report_values.json`, `verification_report.md`, run manifest.
**Position.** NB13 + NB14 → **this** → Chapter 7.
**Run notes.** Executed record — do not re-run. Masking estimates are descriptive dependence diagnostics; all upstream reads are validated against the NB14/NB15 manifests before loading.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip -q install -U pyarrow scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 63.3 MB/s eta 0:00:00


In [3]:
# ==== Imports, Identity, and Contract Paths ====
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import itertools
import copy
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn


NOTEBOOK_NAME = "19_transformer_heldout_interpretation_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"
MODEL_FAMILY = "transformer"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
S2Q_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank/transformer_no_prior"
S2P_DIR = PROJECT_ROOT / "outputs" / "stage2_nonpersonalized_rerank" / "transformer"
FULL_DIR = PROJECT_ROOT / "outputs" / "stage2_personalized_rerank" / "transformer_full"

SOURCE_DIRS = {
    "P2-Q": S2Q_DIR,
    "P2-P": S2P_DIR,
    "Full": FULL_DIR,
}
SOURCE_NOTEBOOKS = {
    "P2-Q": "13a_no_prior_rerank_transformer_herbal.ipynb",
    "P2-P": "13b_01_base_rerank_transformer_herbal.ipynb",
    "Full": "13c_personalized_rerank_transformer_herbal.ipynb",
}

PIPELINE_AGGREGATE_DIR = PROJECT_ROOT / "outputs" / "pipeline_aggregate"
NOTEBOOK14_CANONICAL_METRICS_PATH = (
    PIPELINE_AGGREGATE_DIR / "pipeline_canonical_per_case_metrics.parquet"
)
NOTEBOOK14_MANIFEST_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_manifest.json"
NOTEBOOK15_MANIFEST_PATH = (
    PROJECT_ROOT / "outputs" / "analysis" / "stage_allocation_summary" / "stage_allocation_manifest.json"
)
NOTEBOOK16_MANIFEST_PATH = (
    PROJECT_ROOT / "outputs" / "analysis" / "paired_significance_summary" / "run_manifest.json"
)


OUT_DIR = PROJECT_ROOT / "outputs" / "analysis" / "transformer_interpretation"
if OUT_DIR.parent != PROJECT_ROOT / "outputs" / "analysis":
    raise RuntimeError("Transformer interpretation output directory must remain under category analysis outputs.")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_POOL_DEPTH = 1000
# Depth scope (audit rev2, D19-4 replacement). The reporting headline depth is
# K=1000 and this interpretation layer is computed at the same depth, so the
# interpretation attributes mechanisms to the configuration Ch.6 reports.
# The Notebook 13b export materialises model contracts and held-out
# interpretation parquets at all five depths {100, 300, 500, 700, 1000}
# (audit rev2 SS2.5): depth 1000 here is an alignment choice with the headline,
# not an availability constraint, and K=700 survives only as the appendix
# bounded-reranking *scenario*. The five-depth invariance panel reports depth
# sensitivity separately.
HEADLINE_REPORT_POOL_DEPTH = 1000
INTERPRETATION_DEPTH_SCOPE = (
    "headline_depth_interpretation_pool1000_matches_headline"
)
PRIMARY_K = 5
REPRODUCTION_ATOL = 1e-5
FALLBACK_IDENTITY_ATOL = 1e-7
NATIVE_ALL_PRIOR_SOURCE = "shared_all_prior_model"
FULL_NATIVE_SOURCE = "native_full_self_pool_model"
NO_PRIOR_NATIVE_SOURCE = "no_prior_native"
FALLBACK_SOURCE = "p2q_cold_fallback"  # value written by Notebook 13b/13c cold fallback exports


def native_prediction_sources_for_condition(condition_name):
    if condition_name == "P2-Q":
        return {NO_PRIOR_NATIVE_SOURCE}
    if condition_name == "Full":
        return {NATIVE_ALL_PRIOR_SOURCE, FULL_NATIVE_SOURCE}
    return {NATIVE_ALL_PRIOR_SOURCE}


def observed_native_prediction_source_label(condition_name, frame):
    native_sources = native_prediction_sources_for_condition(condition_name)
    observed = sorted(set(frame["prediction_source"].astype(str)).intersection(native_sources))
    return "|".join(observed) if observed else "|".join(sorted(native_sources))

# ---------------------------------------------------------------------------
# Legacy stage-condition token tolerance.
#
# The canonical stage vocabulary of the 2x2 allocation design is
# {P0, P1-only, P2-Q, P2-P, b02, Full}. Notebooks 13a/13b/13c were sealed before
# that vocabulary was unified and their manifests may still carry the legacy
# "S2-*" spelling. Re-running them is out of scope, so every condition token
# *read from* an upstream artefact is canonicalised before comparison. Tokens
# *written by* this notebook are canonical without exception.
# ---------------------------------------------------------------------------
LEGACY_CONDITION_TOKEN_ALIASES = {
    "S2-Q": "P2-Q",
    "S2-P": "P2-P",
    "s2-q": "P2-Q",
    "s2-p": "P2-P",
}


def canonical_condition_token(value):
    """Map any legacy upstream condition spelling onto the canonical token."""
    text = str(value).strip()
    return LEGACY_CONDITION_TOKEN_ALIASES.get(text, text)


RANDOM_SEED = 20250322

REQUIRED_SEMANTIC_GROUPS = [
    "retrieval_rank",
    "history_depth_recency",
    "functional_preference",
    "query_profile_interaction",
    "candidate_history_interaction",
    "brand_prior",
]
REQUIRED_BRANCH_GROUPS = [
    "candidate_common",
    "functional_prior",
    "brand_prior",
]

OUTPUT_FILES = {
    "source_availability": OUT_DIR / "transformer_source_availability.csv",
    "reconstructed_s2p_manifest": OUT_DIR / "reconstructed_s2p_feature_interpretation_manifest.json",
    "interpretation_scope_note": OUT_DIR / "transformer_interpretation_scope_note.csv",
    "canonical_source_qc": OUT_DIR / "transformer_canonical_source_qc.csv",
    "semantic_group_definition": OUT_DIR / "transformer_semantic_group_definition.csv",
    "branch_group_definition": OUT_DIR / "transformer_branch_group_definition.csv",
    "facet_family_mapping": OUT_DIR / "transformer_facet_family_mapping.csv",
    "reproduction_qc": OUT_DIR / "transformer_fold_reproduction_qc.csv",
    "masking_by_fold": OUT_DIR / "transformer_masking_by_fold.csv",
    "masking_summary": OUT_DIR / "transformer_masking_summary.csv",
    "semantic_group_stability": OUT_DIR / "transformer_semantic_group_stability.csv",
    "semantic_group_fold_rank_correlation": OUT_DIR / "transformer_semantic_group_fold_rank_correlation.csv",
    "primary_semantic_group_summary": OUT_DIR / "transformer_primary_semantic_group_summary_pool1000.csv",
    "primary_facet_family_summary": OUT_DIR / "transformer_primary_facet_family_summary_pool1000.csv",
    "branch_contribution": OUT_DIR / "transformer_branch_contribution_pool1000.csv",
    "cross_model_summary": OUT_DIR / "cross_model_feature_group_summary_transformer.csv",
    "margin_diagnostics": OUT_DIR / "transformer_target_margin_pool1000.csv",
    "top5_taxonomy": OUT_DIR / "transformer_top5_error_taxonomy_pool1000.csv",
    "regime_performance": OUT_DIR / "transformer_regime_performance_pool1000.csv",
    "pool_overlap": OUT_DIR / "transformer_pool_overlap_pool1000.csv",
    "pool_distribution_shift": OUT_DIR / "transformer_pool_distribution_shift_pool1000.csv",
    "checkpoint_diagnostics": OUT_DIR / "transformer_checkpoint_diagnostics.csv",
    "selected_checkpoint_performance": OUT_DIR / "transformer_selected_checkpoint_heldout_performance_pool1000.csv",
    "epoch_diagnostics": OUT_DIR / "transformer_epoch_diagnostics.csv",
    "epoch_diagnostics_availability": OUT_DIR / "transformer_epoch_diagnostics_availability.csv",
    "cold_fallback_qc": OUT_DIR / "transformer_cold_fallback_identity_qc.csv",
    "qc": OUT_DIR / "transformer_interpretation_qc.csv",
    "manifest": OUT_DIR / "run_manifest.json",
}

# Remove stale analysis artifacts before any refresh is attempted.
for _stale_output_path in OUTPUT_FILES.values():
    if Path(_stale_output_path).exists():
        Path(_stale_output_path).unlink()

print("Notebook:", NOTEBOOK_NAME)
print("Category:", CATEGORY_LABEL)
print("Primary pool depth:", PRIMARY_POOL_DEPTH)
print("Output directory:", OUT_DIR)


Notebook: 19_transformer_heldout_interpretation_herbal.ipynb
Category: Herbal Supplements
Primary pool depth: 1000
Output directory: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation


In [4]:
# ==== Validation Helpers ====
def require_columns(frame, required, label):
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(frame, label):
    duplicated = frame.columns[frame.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicate columns: {duplicated}")


def read_json(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def as_path(value, label):
    text = str(value or "").strip()
    if not text:
        raise RuntimeError(f"Missing path in {label}.")
    return Path(text)


def stable_seed(*parts):
    payload = "|".join(map(str, (RANDOM_SEED, *parts))).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**32 - 1)


def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def rank_scores(frame, score_values):
    work = frame[["case_id", "query_id", "candidate_parent_asin", "label"]].copy()
    work["score"] = np.asarray(score_values, dtype=float)
    work["_tie_score"] = pd.to_numeric(
        frame.get("candidate_score_norm_pool", pd.Series(0.0, index=frame.index)),
        errors="coerce",
    ).fillna(0.0).to_numpy(dtype=float)
    work["_tie_rank"] = pd.to_numeric(
        frame.get("candidate_rank", pd.Series(PRIMARY_POOL_DEPTH, index=frame.index)),
        errors="coerce",
    ).fillna(PRIMARY_POOL_DEPTH).to_numpy(dtype=float)
    work["_row_id"] = np.arange(len(work), dtype=np.int64)
    ranked = work.sort_values(
        ["case_id", "score", "_tie_score", "_tie_rank", "candidate_parent_asin"],
        ascending=[True, False, False, True, True],
        kind="mergesort",
    )
    ranked["computed_rank"] = ranked.groupby("case_id", sort=False).cumcount() + 1
    return ranked.sort_values("_row_id", kind="mergesort")["computed_rank"].to_numpy(dtype=np.int32)


def case_metric_frame(frame, score_values, k=PRIMARY_K):
    ranked = frame[["case_id", "query_id", "regime", "label"]].copy()
    ranked["computed_rank"] = rank_scores(frame, score_values)
    rows = []
    for case_id, group in ranked.groupby("case_id", sort=False):
        positives = group.loc[group["label"].astype(int).eq(1), "computed_rank"].astype(int)
        target_rank = int(positives.iloc[0]) if len(positives) else None
        ndcg = 0.0
        if target_rank is not None and target_rank <= int(k):
            ndcg = float(1.0 / np.log2(target_rank + 1.0))
        rows.append({
            "case_id": str(case_id),
            "query_id": str(group["query_id"].iloc[0]),
            "regime": str(group["regime"].iloc[0]).lower(),
            "target_rank": target_rank,
            "NDCG@5": ndcg,
            "HitRate@5": float(target_rank is not None and target_rank <= int(k)),
        })
    return pd.DataFrame(rows)


def metric_scopes(case_metrics):
    scopes = [("overall", "overall", case_metrics)]
    for regime in sorted(case_metrics["regime"].dropna().astype(str).unique()):
        scopes.append(("regime", regime, case_metrics.loc[case_metrics["regime"].eq(regime)]))
    return scopes


def fold_bootstrap_interval(values, seed_parts, n_bootstrap=2000):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if values.size == 0:
        return np.nan, np.nan
    if values.size == 1:
        return float(values[0]), float(values[0])
    rng = np.random.default_rng(stable_seed(*seed_parts))
    samples = rng.choice(values, size=(n_bootstrap, values.size), replace=True).mean(axis=1)
    low, high = np.quantile(samples, [0.025, 0.975])
    return float(low), float(high)


def summarize_masking(masking_frame):
    group_columns = [
        "category", "category_id", "condition_name", "model_family",
        "candidate_pool_depth", "user_scope", "regime", "target_type", "target_name",
    ]
    rows = []
    for keys, group in masking_frame.groupby(group_columns, dropna=False, observed=True):
        values = group["importance_ndcg_at_5_decrease"].to_numpy(dtype=float)
        low, high = fold_bootstrap_interval(values, (*keys, "masking"))
        row = dict(zip(group_columns, keys if isinstance(keys, tuple) else (keys,)))
        row.update({
            "target_feature_count": int(group["target_feature_count"].max()),
            "fold_count": int(group["fold_id"].nunique()),
            "case_count": int(group["case_count"].sum()),
            "baseline_ndcg_at_5_mean": float(group["baseline_ndcg_at_5"].mean()),
            "masked_ndcg_at_5_mean": float(group["masked_ndcg_at_5"].mean()),
            "importance_mean": float(np.mean(values)),
            "importance_std_across_folds": float(np.std(values, ddof=0)),
            "importance_ci_95_low": low,
            "importance_ci_95_high": high,
            "positive_fold_rate": float(np.mean(values > 0)),
            "mean_masked_rank_embedding": bool(group["rank_embedding_masked"].any()),
            "evidence_role": "heldout_group_mean_masking_predictive_dependence",
            "causal_claim": False,
        })
        rows.append(row)
    return pd.DataFrame(rows)


def build_masking_stability(masking_frame, target_type):
    subset = masking_frame.loc[masking_frame["target_type"].eq(target_type)].copy()
    base = [
        "category", "category_id", "condition_name", "model_family",
        "candidate_pool_depth", "user_scope", "regime", "target_type", "target_name",
    ]
    if subset.empty:
        return pd.DataFrame(columns=base)
    subset["fold_rank"] = subset.groupby(
        base[:-1] + ["fold_id"], observed=True
    )["importance_ndcg_at_5_decrease"].rank(method="average", ascending=False)
    return (
        subset.groupby(base, as_index=False, observed=True)
        .agg(
            target_feature_count=("target_feature_count", "max"),
            fold_count=("fold_id", "nunique"),
            importance_mean_across_folds=("importance_ndcg_at_5_decrease", "mean"),
            importance_std_across_folds=("importance_ndcg_at_5_decrease", lambda x: float(np.std(x, ddof=0))),
            importance_min_fold=("importance_ndcg_at_5_decrease", "min"),
            importance_max_fold=("importance_ndcg_at_5_decrease", "max"),
            positive_fold_rate=("importance_ndcg_at_5_decrease", lambda x: float(pd.Series(x).gt(0).mean())),
            mean_fold_rank=("fold_rank", "mean"),
            std_fold_rank=("fold_rank", lambda x: float(np.std(x, ddof=0))),
        )
    )


def build_fold_rank_correlation(masking_frame, target_type):
    subset = masking_frame.loc[masking_frame["target_type"].eq(target_type)].copy()
    groups = [
        "category", "category_id", "condition_name", "model_family",
        "candidate_pool_depth", "user_scope", "regime", "target_type",
    ]
    rows = []
    for keys, scope_df in subset.groupby(groups, dropna=False, observed=True):
        fold_values = scope_df.pivot_table(
            index="target_name", columns="fold_id",
            values="importance_ndcg_at_5_decrease", aggfunc="mean",
        )
        for left, right in itertools.combinations(fold_values.columns.tolist(), 2):
            paired = fold_values[[left, right]].dropna()
            rho = paired[left].corr(paired[right], method="spearman") if len(paired) >= 2 else np.nan
            row = dict(zip(groups, keys if isinstance(keys, tuple) else (keys,)))
            row.update({
                "left_fold_id": int(left),
                "right_fold_id": int(right),
                "compared_target_count": int(len(paired)),
                "spearman_rank_correlation": float(rho) if pd.notna(rho) else np.nan,
            })
            rows.append(row)
    return pd.DataFrame(rows)




def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def validate_refreshed_analysis_chain():
    required = [
        NOTEBOOK14_CANONICAL_METRICS_PATH,
        NOTEBOOK14_MANIFEST_PATH,
        NOTEBOOK15_MANIFEST_PATH,
        NOTEBOOK16_MANIFEST_PATH,
    ]
    missing = [str(path) for path in required if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(
            "Notebook 19 requires refreshed Notebook 14-16 artifacts in the order "
            "13c -> 14 -> 15 -> 16 -> 19. Missing: " + json.dumps(missing)
        )
    nb14 = read_json(NOTEBOOK14_MANIFEST_PATH, "Notebook 14 pipeline manifest")
    nb15 = read_json(NOTEBOOK15_MANIFEST_PATH, "Notebook 15 stage-allocation manifest")
    nb16 = read_json(NOTEBOOK16_MANIFEST_PATH, "Notebook 16 paired-inference manifest")
    if nb14.get("run_status") != "SUCCESS":
        raise RuntimeError("Notebook 14 is not sealed with run_status=SUCCESS.")
    if nb14.get("ready_for_downstream") is not True or nb14.get("full_transfer_readiness") is not True:
        raise RuntimeError("Notebook 14 is not ready after final Full-transfer validation.")
    canonical_sha = file_sha256(NOTEBOOK14_CANONICAL_METRICS_PATH)
    nb14_sha = nb14.get("canonical_raw_sha256")
    if nb14_sha and nb14_sha != canonical_sha:
        raise RuntimeError("Notebook 14 canonical metric SHA differs from its manifest.")
    if nb15.get("category_id") != CATEGORY_ID or nb15.get("ready_for_thesis_reporting") is not True:
        raise RuntimeError("Notebook 15 is absent, stale, or not ready for thesis reporting.")
    if nb15.get("notebook14_canonical_raw_sha256") != canonical_sha:
        raise RuntimeError("Notebook 15 was not regenerated from the current Notebook 14 canonical artifact.")
    if nb16.get("category_id") != CATEGORY_ID:
        raise RuntimeError("Notebook 16 category does not match Notebook 19.")
    nb16_inputs = nb16.get("input_sha256", {})
    if nb16_inputs.get("canonical_per_case_metrics") != canonical_sha:
        raise RuntimeError("Notebook 16 was not regenerated from the current Notebook 14 canonical artifact.")
    validation = nb16.get("validation_results", {})
    required_flags = [
        "notebook14_paired_universe_qc_passed",
        "authoritative_inputs_only",
        "holm_adjustment_applied_to_locked_eight_principal_tests",
    ]
    failed = [flag for flag in required_flags if validation.get(flag) is not True]
    if failed:
        raise RuntimeError(f"Notebook 16 locked inference validation failed: {failed}")
    return {
        "execution_order": ["13c", "14", "15", "16", "19"],
        "notebook14_manifest": str(NOTEBOOK14_MANIFEST_PATH),
        "notebook15_manifest": str(NOTEBOOK15_MANIFEST_PATH),
        "notebook16_manifest": str(NOTEBOOK16_MANIFEST_PATH),
        "notebook14_canonical_sha256": canonical_sha,
        "notebook14_ready": True,
        "notebook15_ready": True,
        "notebook16_ready": True,
    }


class S2PInterpretationUnavailable(RuntimeError):
    """Expected absence or inconsistency of authoritative P2-P interpretation artifacts."""


def _s2p_read_json(path, label):
    try:
        return read_json(path, label)
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        raise S2PInterpretationUnavailable(f"{label}: {exc}") from exc


def _s2p_as_path(value, label):
    try:
        return as_path(value, label)
    except RuntimeError as exc:
        raise S2PInterpretationUnavailable(f"{label}: {exc}") from exc


def _s2p_require_columns(frame, required, label):
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise S2PInterpretationUnavailable(
            f"{label} is missing required columns: {missing}"
        )


def reconstruct_s2p_interpretation_manifest(source_dir: Path, run_manifest: dict):
    """Path A: reconstruct metadata only after all authoritative 13b artifacts agree."""
    source_dir = Path(source_dir)
    canonical_path = Path(OUTPUT_FILES["reconstructed_s2p_manifest"])
    # SC-2 pre-write containment check for the reconstructed manifest.
    try:
        canonical_path.resolve().relative_to(OUT_DIR.resolve())
    except ValueError as exc:
        raise RuntimeError(
            "Reconstructed P2-P interpretation manifest is outside Notebook 19 OUT_DIR."
        ) from exc
    metadata_candidates = [
        source_dir / "feature_interpretation_manifest.json",
        source_dir / "feature_interpretation_manifest.json.disabled_category_mismatch",
        source_dir / "feature_interpretation_manifest.disabled_category_mismatch.json",
    ]
    template_path = next((path for path in metadata_candidates if path.exists()), None)
    template = _s2p_read_json(template_path, "P2-P interpretation metadata source") if template_path else {}

    feature_manifest_path = _s2p_as_path(
        run_manifest.get("output_paths", {}).get(
            "feature_manifest", run_manifest.get("feature_manifest_path", source_dir / "feature_manifest.json")
        ),
        "P2-P feature manifest",
    )
    feature_manifest = _s2p_read_json(feature_manifest_path, "P2-P feature manifest")
    _feature_manifest_category = feature_manifest.get("category_id")
    if _feature_manifest_category is not None and str(_feature_manifest_category) != CATEGORY_ID:
        raise S2PInterpretationUnavailable("P2-P feature manifest category mismatch.")
    feature_names = list(
        feature_manifest.get("shared_all_prior_feature_columns")
        or feature_manifest.get("model_features")
        or feature_manifest.get("feature_names")
        or []
    )
    if len(feature_names) != 55 or len(feature_names) != len(set(feature_names)):
        raise S2PInterpretationUnavailable("P2-P reconstruction requires the exact 55-feature primary registry.")
    exact_item = {
        "user_item_seen_strength", "user_item_recency_days", "user_item_recent_count_180d",
    }
    if set(feature_names).intersection(exact_item):
        raise S2PInterpretationUnavailable("P2-P reconstruction source includes previously-reviewed-item variables.")
    if bool(feature_manifest.get("exact_item_familiarity_enabled", False)):
        raise S2PInterpretationUnavailable("P2-P primary feature manifest enables exact-item familiarity.")

    mapping_path = Path(str(template.get("feature_group_mapping_path", source_dir / "feature_group_mapping.csv")))
    assignment_path = Path(str(template.get(
        "fold_assignment_path", source_dir / "feature_interpretation_fold_assignments.parquet"
    )))
    if not mapping_path.exists() or not assignment_path.exists():
        raise S2PInterpretationUnavailable("P2-P feature mapping or held-out fold assignment is missing.")
    mapping_df = pd.read_csv(mapping_path)
    _s2p_require_columns(mapping_df, ["feature", "interpretation_group"], "P2-P feature mapping")
    if mapping_df["feature"].astype(str).tolist() != feature_names:
        raise S2PInterpretationUnavailable("P2-P feature-group mapping order differs from the 55-feature registry.")

    contract_rows = []
    checkpoint_rows = []
    heldout_rows = []
    shared_settings = None
    shared_dtypes = None
    for depth in [100, 300, 500, 700, 1000]:
        contract_path = source_dir / f"shared_all_prior_transformer_pool_{depth}_contract.json"
        heldout_path = source_dir / f"heldout_feature_interpretation_pool{depth}.parquet"
        if not contract_path.exists() or not heldout_path.exists():
            raise S2PInterpretationUnavailable(
                f"Missing P2-P contract or held-out interpretation pool at depth={depth}."
            )
        contract = _s2p_read_json(contract_path, f"P2-P depth-{depth} model contract")
        if contract.get("category_id") != CATEGORY_ID or int(contract.get("pool_depth", -1)) != depth:
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} contract category/depth mismatch.")
        if contract.get("feature_columns") != feature_names:
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} feature order mismatch.")
        dtypes = contract.get("feature_dtypes", {})
        if set(dtypes) != set(feature_names):
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} dtype contract is incomplete.")
        if shared_dtypes is None:
            shared_dtypes = dtypes
        elif dtypes != shared_dtypes:
            raise S2PInterpretationUnavailable("P2-P feature dtypes change across depths.")
        settings = contract.get("settings", {})
        if settings.get("objective") != "listwise_cross_entropy":
            raise S2PInterpretationUnavailable("P2-P objective is not listwise cross-entropy.")
        if settings.get("checkpoint_selection_metric") != "validation_ndcg_at_5":
            raise S2PInterpretationUnavailable("P2-P checkpoint selection metric is stale or incorrect.")
        expected_rule = (
            "maximize_validation_ndcg_at_5_then_minimize_validation_listwise_ce_then_earliest_epoch"
        )
        if settings.get("checkpoint_selection_rule") != expected_rule:
            raise S2PInterpretationUnavailable("P2-P checkpoint-selection rule mismatch.")
        if shared_settings is None:
            shared_settings = settings
        elif settings != shared_settings:
            raise S2PInterpretationUnavailable("P2-P Transformer settings change across candidate depths.")
        heldout_schema = set(pq.ParquetFile(heldout_path).schema.names)
        heldout_required = {
            "condition_name", "category_id", "pool_depth", "case_id", "query_id",
            "candidate_parent_asin", "candidate_source_slug", "candidate_rank",
            "label", "fold_id", "oof_prediction", *feature_names,
        }
        missing_heldout = sorted(heldout_required.difference(heldout_schema))
        if missing_heldout:
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} held-out artifact is incomplete: {missing_heldout}")
        fold_assignment_path = Path(str(contract.get("fold_assignment_path", "")))
        if not fold_assignment_path.exists():
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} fold assignment is missing.")
        fold_contracts = contract.get("folds", [])
        if len(fold_contracts) != 5 or {int(row["fold"]) for row in fold_contracts} != {1,2,3,4,5}:
            raise S2PInterpretationUnavailable(f"P2-P depth-{depth} does not contain five fold contracts.")
        for fold_info in fold_contracts:
            checkpoint_path = Path(str(fold_info.get("model_path", "")))
            if not checkpoint_path.exists():
                raise S2PInterpretationUnavailable(f"Missing P2-P checkpoint: {checkpoint_path}")
            observed_sha = file_sha256(checkpoint_path)
            if observed_sha != str(fold_info.get("model_sha256", "")):
                raise S2PInterpretationUnavailable(f"P2-P checkpoint SHA mismatch: {checkpoint_path}")
            checkpoint_rows.append({
                "pool_depth": depth,
                "fold_id": int(fold_info["fold"]),
                "path": str(checkpoint_path),
                "sha256": observed_sha,
                "preprocessing_sha256": str(fold_info.get("preprocessing_sha256", "")),
                "fold_seed": fold_info.get("fold_seed"),
                "fallback_used": False,
            })
        contract_rows.append({
            "pool_depth": depth,
            "path": str(contract_path),
            "fold_assignment_path": str(fold_assignment_path),
            "global_fallback_used": bool(contract.get("global_fallback_used", False)),
            "contract_sha256": file_sha256(contract_path),
        })
        heldout_rows.append({
            "pool_depth": depth,
            "path": str(heldout_path),
            "row_count": int(pq.ParquetFile(heldout_path).metadata.num_rows),
            "validation": "schema_and_parquet_metadata_only_no_full_file_hash",
        })

    reconstructed = {
        "artifact_version": "transformer_heldout_structured_feature_interpretation_v1",
        "condition_name": "P2-P",
        "category_id": CATEGORY_ID,
        "model_family": MODEL_FAMILY,
        "task_scope": "query_conditioned_next_novel_item_ranking",
        "specification_role": "primary",
        "primary_registry_policy_version": feature_manifest.get(
            "primary_registry_policy_version", "benchmark_primary_novel_item_registry_v1"
        ),
        "exact_item_familiarity_enabled": False,
        "exact_item_familiarity_feature_columns": [],
        "model_source_condition": "P2-P",
        "model_configuration": shared_settings,
        "tokenizer_text_encoder_contract": template.get("tokenizer_text_encoder_contract", {}),
        "structured_feature_injection_contract": template.get("structured_feature_injection_contract", {}),
        "attention_interpretation_policy": template.get(
            "attention_interpretation_policy", "attention_weights_not_exported_or_treated_as_explanations"
        ),
        "feature_names": feature_names,
        "structured_feature_names": feature_names,
        "structured_feature_dtypes": shared_dtypes,
        "structured_feature_count": 55,
        "structured_feature_preprocessing_contract": template.get(
            "structured_feature_preprocessing_contract",
            next(iter([_s2p_read_json(Path(row["path"]), "P2-P contract").get(
                "structured_feature_preprocessing_contract", {}
            ) for row in contract_rows]), {}),
        ),
        "missing_value_policy": template.get("missing_value_policy", "contract_defined"),
        "feature_group_mapping_path": str(mapping_path),
        "fold_assignment_path": str(assignment_path),
        "fold_assignment_row_count": int(pq.ParquetFile(assignment_path).metadata.num_rows),
        "candidate_feature_oof_artifacts": heldout_rows,
        "model_contracts": contract_rows,
        "fold_model_checkpoints": checkpoint_rows,
        "oof_prediction_column": "oof_prediction",
        "label_column": "label",
        "case_id_column": "case_id",
        "candidate_id_column": "candidate_parent_asin",
        "fold_id_column": "fold_id",
        "random_seed": 42,
        "candidate_source_contract": template.get(
            "candidate_source_contract", run_manifest.get("candidate_source_qc", {})
        ),
        "brand_all_prior_flags": template.get("brand_all_prior_flags", {}),
        "s2p_full_feature_schema_equal": True,
        "s2p_full_model_settings_equal": True,
        "s2p_full_fold_assignment_equal": None,
        "disabled_category_mismatch": False,
        "reconstruction_mode": "path_A_metadata_only_from_authoritative_13b_artifacts",
        "reconstruction_source_feature_manifest": str(feature_manifest_path),
        "reconstruction_source_feature_manifest_sha256": file_sha256(feature_manifest_path),
        "reconstruction_template_path": str(template_path) if template_path else "",
        "reconstruction_template_sha256": file_sha256(template_path) if template_path else "",
        "reconstruction_source_run_manifest_path": str(source_dir / "run_manifest.json"),
        "reconstruction_source_run_manifest_sha256": file_sha256(source_dir / "run_manifest.json"),
        "reconstruction_source_feature_group_mapping_path": str(mapping_path),
        "reconstruction_source_feature_group_mapping_sha256": file_sha256(mapping_path),
        "reconstruction_source_fold_assignment_path": str(assignment_path),
        "reconstruction_source_fold_assignment_sha256": file_sha256(assignment_path),
        "reconstruction_output_path": str(canonical_path),
        "upstream_source_directory": str(source_dir),
        "upstream_artifacts_modified": False,
        "reconstructed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    canonical_path.parent.mkdir(parents=True, exist_ok=True)
    canonical_path.write_text(json.dumps(reconstructed, ensure_ascii=False, indent=2), encoding="utf-8")
    return reconstructed, canonical_path, "path_A_authoritative_metadata_reconstruction", ""


# Notebook 14 is the sole canonical performance authority.
NOTEBOOK14_REQUIRED_COLUMNS = [
    "case_id", "query_id", "user_id", "regime",
    "stage_condition", "reranker_method", "method_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff", "metric_value",
    "target_exposed", "target_rank", "qchs_profile_available",
    "profile_fallback_flag", "candidate_source", "source_notebook",
    "shared_stage1_baseline", "shared_baseline_repeated_for_display",
]


def load_notebook14_transformer_metrics() -> tuple[pd.DataFrame, dict, Path, Path, dict]:
    analysis_chain_preflight = validate_refreshed_analysis_chain()
    metric_path = NOTEBOOK14_CANONICAL_METRICS_PATH
    manifest_path = NOTEBOOK14_MANIFEST_PATH
    missing = [str(path) for path in [metric_path, manifest_path] if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing final Notebook 14 canonical artifacts. Run Notebook 14 after both 13c notebooks: "
            + json.dumps(missing, ensure_ascii=False)
        )

    manifest = read_json(manifest_path, "Notebook 14 pipeline manifest")
    if manifest.get("run_status") != "SUCCESS":
        raise RuntimeError("Notebook 14 is not sealed with run_status=SUCCESS.")
    manifest_outputs = manifest.get("output_paths", {})
    manifest_canonical_raw = str(manifest_outputs.get("canonical_raw", "")).strip()
    if not manifest_canonical_raw or Path(manifest_canonical_raw).name != metric_path.name:
        raise RuntimeError("Notebook 14 manifest canonical_raw path disagrees with the locked artifact.")

    raw = pd.read_parquet(metric_path).copy()
    require_columns(raw, NOTEBOOK14_REQUIRED_COLUMNS, "Notebook 14 canonical per-case metrics")
    require_unique_columns(raw, "Notebook 14 canonical per-case metrics")
    canonical_key = [
        "case_id", "stage_condition", "reranker_method",
        "candidate_pool_depth", "metric_name", "metric_cutoff",
    ]
    if raw.duplicated(canonical_key).any():
        raise RuntimeError("Notebook 14 canonical per-case metrics contain duplicate inferential keys.")
    if raw["shared_baseline_repeated_for_display"].astype(bool).any():
        raise RuntimeError("Repeated presentation-only rows entered Notebook 14 canonical metrics.")

    transformer = raw.loc[
        raw["method_family"].astype(str).eq("transformer")
        & pd.to_numeric(raw["candidate_pool_depth"], errors="raise")
        .astype(int).eq(PRIMARY_POOL_DEPTH)
        & raw["stage_condition"].astype(str).isin(["P2-Q", "P2-P", "Full"])
    ].copy()
    if transformer.empty:
        raise RuntimeError("Notebook 14 contains no Transformer rows at the depth-1000 headline interface.")

    ndcg = transformer.loc[
        transformer["metric_name"].astype(str).eq("NDCG")
        & pd.to_numeric(transformer["metric_cutoff"], errors="raise").astype(int).eq(PRIMARY_K),
        ["case_id", "query_id", "user_id", "regime", "stage_condition", "metric_value"],
    ].copy()
    hit = transformer.loc[
        transformer["metric_name"].astype(str).eq("HitRate")
        & pd.to_numeric(transformer["metric_cutoff"], errors="raise").astype(int).eq(PRIMARY_K),
        ["case_id", "query_id", "user_id", "regime", "stage_condition", "metric_value"],
    ].copy()
    if ndcg.empty or hit.empty:
        raise RuntimeError("Notebook 14 lacks Transformer NDCG@5 or HitRate@5 at depth 1000.")

    merge_key = ["case_id", "query_id", "user_id", "regime", "stage_condition"]
    if ndcg.duplicated(merge_key).any() or hit.duplicated(merge_key).any():
        raise RuntimeError("Notebook 14 Transformer headline metrics are duplicated at the case-condition grain.")
    canonical = ndcg.rename(columns={"metric_value": "NDCG@5"}).merge(
        hit.rename(columns={"metric_value": "HitRate@5"}),
        on=merge_key,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    if not canonical["_merge"].eq("both").all():
        raise RuntimeError("Notebook 14 NDCG@5 and HitRate@5 case universes do not match.")
    canonical = canonical.drop(columns="_merge").rename(
        columns={"stage_condition": "condition_name"}
    )
    canonical["case_id"] = canonical["case_id"].astype(str)
    canonical["query_id"] = canonical["query_id"].astype(str)
    canonical["user_id"] = canonical["user_id"].astype(str)
    canonical["regime"] = canonical["regime"].astype(str).str.lower()
    canonical["condition_name"] = canonical["condition_name"].astype(str)
    canonical["NDCG@5"] = pd.to_numeric(canonical["NDCG@5"], errors="raise").astype(float)
    canonical["HitRate@5"] = pd.to_numeric(canonical["HitRate@5"], errors="raise").astype(float)

    if not np.isfinite(canonical[["NDCG@5", "HitRate@5"]].to_numpy(dtype=float)).all():
        raise RuntimeError("Notebook 14 Transformer headline metrics contain non-finite values.")
    if set(canonical["condition_name"]) != {"P2-Q", "P2-P", "Full"}:
        raise RuntimeError("Notebook 14 Transformer conditions must be exactly Base, RankP, and Full.")
    if canonical.duplicated(["condition_name", "case_id"]).any():
        raise RuntimeError("Notebook 14 Transformer canonical metrics duplicate condition-case keys.")

    case_sets = {
        condition: set(group["case_id"])
        for condition, group in canonical.groupby("condition_name", sort=True)
    }
    if len({frozenset(values) for values in case_sets.values()}) != 1:
        raise RuntimeError(
            "Notebook 14 Transformer condition case universes differ: "
            + json.dumps({key: len(value) for key, value in case_sets.items()}, sort_keys=True)
        )
    return canonical, manifest, metric_path, manifest_path, analysis_chain_preflight


In [5]:
# ==== Load Run Manifests and Interpretation Availability ====
run_manifests = {}
reranked_paths = {}
interpretation_manifests = {}
interpretation_manifest_paths = {}
availability_rows = []
s2p_interpretation_mode = "not_evaluated"
s2p_interpretation_reason = ""

for condition_name, source_dir in SOURCE_DIRS.items():
    interpretation_manifest = None
    run_manifest_path = source_dir / "run_manifest.json"
    run_manifest = read_json(run_manifest_path, f"{condition_name} run manifest")
    if run_manifest.get("category_id") != CATEGORY_ID:
        raise RuntimeError(f"{condition_name} run-manifest category mismatch.")
    expected_candidate_role = (
        "personalized_retrieval" if condition_name == "Full" else "baseline_query_only"
    )
    if run_manifest.get("candidate_pool_role") != expected_candidate_role:
        raise RuntimeError(
            f"{condition_name} candidate-source lineage mismatch: "
            f"expected={expected_candidate_role}, observed={run_manifest.get('candidate_pool_role')}"
        )
    prior_qc = run_manifest.get("prior_history_source_qc", {})
    if not bool(prior_qc.get("strict_temporal_validation_passed", False)):
        raise RuntimeError(f"{condition_name} upstream strict pre-target temporal QC did not pass.")
    if run_manifest.get("checkpoint_selection_metric") != "validation_ndcg_at_5":
        raise RuntimeError(
            f"{condition_name} checkpoint selection metric must be validation_ndcg_at_5, "
            f"observed={run_manifest.get('checkpoint_selection_metric')}"
        )
    if condition_name == "Full":
        # Legacy 13c manifests do not carry the downstream seal fields added later.
        # Accept either the newer sealed load-and-score contract or the existing trained-Full contract.
        _full_role = str(run_manifest.get("shared_all_prior_model_role", "train_and_export"))
        _new_sealed_full = (
            run_manifest.get("run_status") == "SUCCESS"
            and run_manifest.get("ready_for_downstream") is True
            and int(run_manifest.get("optimizer_step_count", -1)) == 0
            and bool(run_manifest.get("model_training_performed", True)) is False
            and bool(run_manifest.get("loaded_checkpoint_sha_verified_against_13b_export", False))
            and bool(run_manifest.get("loaded_standardizer_sha_verified_against_13b_export", False))
        )
        _legacy_trained_full = _full_role == "train_and_export"
        if not (_new_sealed_full or _legacy_trained_full):
            raise RuntimeError(
                "Full 13c manifest is neither a sealed load-and-score run nor the legacy trained-Full run."
            )
    run_manifests[condition_name] = run_manifest
    output_paths = run_manifest.get("output_paths", {})
    leakage_qc_path = as_path(
        output_paths.get("feature_leakage_qc", run_manifest.get("feature_leakage_qc_path")),
        f"{condition_name} feature leakage QC",
    )
    if not leakage_qc_path.exists():
        raise FileNotFoundError(f"Missing {condition_name} feature leakage QC: {leakage_qc_path}")
    leakage_qc = pd.read_csv(leakage_qc_path)
    require_columns(leakage_qc, ["check_name", "check_passed"], f"{condition_name} feature leakage QC")
    leakage_passed = leakage_qc["check_passed"].astype(str).str.lower().isin({"true", "1"})
    if not leakage_passed.all():
        raise RuntimeError(f"{condition_name} upstream feature leakage QC contains failures.")
    reranked_path = as_path(
        output_paths.get("reranked_candidates", source_dir / "reranked_candidates.parquet"),
        f"{condition_name} reranked candidates",
    )
    if not reranked_path.exists():
        raise FileNotFoundError(f"Missing {condition_name} reranked candidate export: {reranked_path}")
    reranked_paths[condition_name] = reranked_path

    interpretation_path = Path(str(output_paths.get(
        "feature_interpretation_manifest", source_dir / "feature_interpretation_manifest.json"
    )))
    available = interpretation_path.exists()
    path_mode = "existing_valid_manifest"
    scope_reason = ""
    if condition_name == "P2-P" and not available:
        try:
            interpretation_manifest, interpretation_path, path_mode, scope_reason = (
                reconstruct_s2p_interpretation_manifest(source_dir, run_manifest)
            )
            available = True
        except S2PInterpretationUnavailable as exc:
            available = False
            s2p_interpretation_mode = "path_B_explicit_scope_reduction"
            s2p_interpretation_reason = f"{type(exc).__name__}: {exc}"
            path_mode = s2p_interpretation_mode
            scope_reason = s2p_interpretation_reason
    if condition_name in {"P2-Q", "Full"} and not available:
        raise FileNotFoundError(
            f"Required {condition_name} held-out interpretation manifest is missing: {interpretation_path}"
        )
    if available:
        if interpretation_manifest is None:
            interpretation_manifest = read_json(
                interpretation_path, f"{condition_name} interpretation manifest"
            )
        if interpretation_manifest.get("category_id") != CATEGORY_ID:
            raise RuntimeError(f"{condition_name} interpretation-manifest category mismatch.")
        if interpretation_manifest.get("model_family") != MODEL_FAMILY:
            raise RuntimeError(f"{condition_name} interpretation-manifest model-family mismatch.")
        if canonical_condition_token(
            interpretation_manifest.get("condition_name")
        ) != condition_name:
            raise RuntimeError(f"{condition_name} interpretation-manifest condition mismatch.")
        if condition_name == "Full":
            _full_source_condition = canonical_condition_token(
                interpretation_manifest.get("model_source_condition")
            )
            _full_role = str(run_manifest.get("shared_all_prior_model_role", "train_and_export"))
            _valid_full_source = (
                (_full_role == "load_and_score" and _full_source_condition == "P2-P")
                or (_full_role == "train_and_export" and _full_source_condition == "Full")
            )
            if not _valid_full_source:
                raise RuntimeError(
                    "Full model source condition must be P2-P for load-and-score or Full for legacy trained-Full."
                )
        if condition_name == "Full":
            _full_role = str(run_manifest.get("shared_all_prior_model_role", "train_and_export"))
            _disabled_mismatch = interpretation_manifest.get("disabled_category_mismatch")
            if _full_role == "load_and_score" and _disabled_mismatch is not False:
                raise RuntimeError(
                    "Full load-and-score feature_interpretation_manifest.json must set disabled_category_mismatch = false."
                )
            if _full_role == "train_and_export" and _disabled_mismatch not in {False, True, None}:
                raise RuntimeError(
                    "Full legacy trained-Full feature_interpretation_manifest.json has an invalid disabled_category_mismatch value."
                )
        if interpretation_manifest.get("model_configuration", {}).get(
            "checkpoint_selection_metric"
        ) != "validation_ndcg_at_5":
            raise RuntimeError(f"{condition_name} interpretation manifest contains stale checkpoint metadata.")
        interpretation_manifests[condition_name] = interpretation_manifest
        interpretation_manifest_paths[condition_name] = interpretation_path
        if condition_name == "P2-P":
            s2p_interpretation_mode = (
                path_mode if path_mode != "existing_valid_manifest"
                else "path_A_existing_authoritative_manifest"
            )
            s2p_interpretation_reason = ""
    availability_rows.append({
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "condition_name": condition_name,
        "run_manifest_path": str(run_manifest_path),
        "reranked_candidates_path": str(reranked_path),
        "interpretation_manifest_path": str(interpretation_path),
        "interpretation_manifest_required_for_chapter7": True,
        "interpretation_manifest_available": bool(available),
        "interpretation_path_mode": path_mode,
        "scope_reduction_reason": scope_reason,
        "masking_coverage_included": bool(available),
        "analysis_policy": (
            "reproduction_and_masking"
            if available
            else "explicit_scope_reduction_no_s2p_masking_no_proxy_invented"
        ),
    })
    if "interpretation_manifest" in locals():
        del interpretation_manifest

source_availability_df = pd.DataFrame(availability_rows)
if s2p_interpretation_mode == "not_evaluated":
    raise RuntimeError("RankP interpretation availability was not evaluated.")
_s2p_available = "P2-P" in interpretation_manifests
interpretation_scope_note_df = pd.DataFrame([{
    "category": CATEGORY_LABEL,
    "category_id": CATEGORY_ID,
    "condition_name": "P2-P",
    "interpretation_path": s2p_interpretation_mode,
    "s2p_masking_available": bool(_s2p_available),
    "reason": s2p_interpretation_reason,
    "thesis_facing_scope_note": (
        "P2-P held-out masking was reconstructed and validated from authoritative 13b metadata and artifacts."
        if _s2p_available
        else (
            "P2-P held-out masking is unavailable because the authoritative 13b metadata could not be "
            "reconstructed without disagreement. Chapter 7 must restrict Transformer masking evidence "
            "to conditions with valid manifests and must not imply complete P2-P coverage."
        )
    ),
}])

(
    notebook14_canonical_metrics_df,
    notebook14_canonical_manifest,
    notebook14_canonical_metrics_path,
    notebook14_canonical_manifest_path,
    analysis_chain_preflight,
) = load_notebook14_transformer_metrics()

canonical_source_qc_df = pd.DataFrame([{
    "category": CATEGORY_LABEL,
    "category_id": CATEGORY_ID,
    "canonical_metrics_path": str(notebook14_canonical_metrics_path),
    "canonical_manifest_path": str(notebook14_canonical_manifest_path),
    "row_count": int(len(notebook14_canonical_metrics_df)),
    "case_count_s2q": int(
        notebook14_canonical_metrics_df.loc[
            notebook14_canonical_metrics_df["condition_name"].eq("P2-Q"), "case_id"
        ].nunique()
    ),
    "case_count_s2p": int(
        notebook14_canonical_metrics_df.loc[
            notebook14_canonical_metrics_df["condition_name"].eq("P2-P"), "case_id"
        ].nunique()
    ),
    "case_count_full": int(
        notebook14_canonical_metrics_df.loc[
            notebook14_canonical_metrics_df["condition_name"].eq("Full"), "case_id"
        ].nunique()
    ),
    "primary_metric": "NDCG@5",
    "candidate_pool_depth": int(PRIMARY_POOL_DEPTH),
    "performance_authority": "Notebook14",
    "canonical_source_passed": True,
}])


def feature_groups_for_condition(condition_name, manifest):
    feature_names = list(manifest.get("structured_feature_names", []))
    if not feature_names or len(feature_names) != len(set(feature_names)):
        raise RuntimeError(f"{condition_name} structured feature names are empty or duplicated.")
    exact_item_familiarity_features = {
        "user_item_seen_strength",
        "user_item_recency_days",
        "user_item_recent_count_180d",
    }
    exact_item_in_primary = sorted(set(feature_names).intersection(exact_item_familiarity_features))
    if exact_item_in_primary:
        raise RuntimeError(
            f"{condition_name} primary Transformer interpretation includes "
            f"previously-reviewed-item diagnostics: {exact_item_in_primary}"
        )
    if manifest.get("specification_role", "primary") != "primary":
        raise RuntimeError(f"{condition_name} interpretation requires specification_role=primary.")
    if bool(manifest.get("exact_item_familiarity_enabled", False)):
        raise RuntimeError(f"{condition_name} primary interpretation cannot enable exact-item familiarity.")
    dtype_contract = manifest.get("structured_feature_dtypes", {})
    if set(dtype_contract) != set(feature_names):
        raise RuntimeError(f"{condition_name} structured feature dtype contract is incomplete.")

    mapping_path = as_path(manifest.get("feature_group_mapping_path"), "feature group mapping")
    mapping_df = pd.read_csv(mapping_path)
    require_columns(mapping_df, ["feature", "interpretation_group"], f"{condition_name} feature mapping")
    if mapping_df["feature"].astype(str).tolist() != feature_names:
        raise RuntimeError(f"{condition_name} feature mapping order differs from the model contract.")
    upstream_group = dict(zip(mapping_df["feature"].astype(str), mapping_df["interpretation_group"].astype(str)))

    forbidden_exact = {
        "case_id", "query_id", "user_id", "target_parent_asin", "gt_item_id",
        "candidate_parent_asin", "candidate_item_id", "is_gt", "label",
        "target_timestamp_ms", "prior_timestamp_ms", "review_timestamp_ms",
        "query_text", "review_text", "target_review_text", "review_body",
    }
    forbidden = sorted(
        feature for feature in feature_names
        if feature in forbidden_exact or feature.endswith("_id")
    )
    brand_query_features = sorted(
        feature for feature in feature_names
        if feature.startswith("qmatch__brand") or "query_brand" in feature
    )
    if forbidden or brand_query_features:
        raise RuntimeError(
            f"{condition_name} target/future/identifier or query-brand leakage features detected: "
            f"{forbidden + brand_query_features}"
        )

    brand_flags = manifest.get("brand_all_prior_flags", {})
    brand_prior = {
        feature for feature in brand_flags.get("brand_affinity_feature_columns", [])
        if feature in feature_names and feature != "candidate_brand_present"
    }
    brand_prior.update({
        feature for feature in feature_names
        if (
            feature.startswith("candidate_brand_prior_")
            or feature in {
                "candidate_brand_last_interaction_age_days",
                "candidate_brand_recent_interaction_count_180d",
                "user_prior_unique_brand_count",
                "user_prior_brand_entropy_norm",
            }
        )
    })
    candidate_history = {
        feature for feature in feature_names
        if feature.startswith(("ucountlog__", "useen__"))
    }
    functional_preference = {
        feature for feature in feature_names
        if feature.startswith("uaff__")
        or feature in {"user_entropy_norm_mean", "user_top_share_mean", "user_item_affinity"}
    }
    query_profile = {
        feature for feature in feature_names
        if feature.startswith(("qprofile__", "query_profile__", "query_history__", "qhist__"))
    }
    history_depth_recency = {
        feature for feature in feature_names
        if feature in {
            "prior_history_count", "prior_unique_item_count", "prior_history_n",
            "prior_review_n", "prior_review_n_log1p", "prior_item_n", "prior_item_n_log1p",
            "user_last_interaction_gap_days",
        }
        or (
            feature not in exact_item_familiarity_features
            and feature.startswith("user_")
            and feature.endswith(("_recency_days", "_recent_count_180d", "_interaction_gap_days"))
        )
    }
    history_depth_recency -= brand_prior | candidate_history | functional_preference | query_profile
    retrieval_rank = {
        feature for feature in feature_names
        if upstream_group.get(feature) == "retrieval"
        or feature.startswith("candidate_rank")
        or feature in {"candidate_score", "candidate_score_norm_pool"}
    }

    semantic_map = {
        "retrieval_rank": sorted(retrieval_rank),
        "history_depth_recency": sorted(history_depth_recency),
        "functional_preference": sorted(functional_preference),
        "query_profile_interaction": sorted(query_profile),
        "candidate_history_interaction": sorted(candidate_history),
        "brand_prior": sorted(brand_prior),
    }
    memberships = [feature for values in semantic_map.values() for feature in values]
    duplicated_membership = sorted({feature for feature in memberships if memberships.count(feature) > 1})
    if duplicated_membership:
        raise RuntimeError(f"{condition_name} semantic groups overlap: {duplicated_membership}")
    if condition_name != "P2-Q":
        required_nonempty = set(REQUIRED_SEMANTIC_GROUPS) - {"query_profile_interaction"}
        empty_required = sorted(group for group in required_nonempty if not semantic_map[group])
        if empty_required:
            raise RuntimeError(f"{condition_name} required semantic groups are empty: {empty_required}")

    functional_prior = sorted(
        set(history_depth_recency)
        | set(functional_preference)
        | set(query_profile)
        | set(candidate_history)
    )
    branch_map = {
        "functional_prior": functional_prior,
        "brand_prior": sorted(brand_prior),
    }
    prior_union = set(branch_map["functional_prior"]) | set(branch_map["brand_prior"])
    branch_map["candidate_common"] = sorted(set(feature_names) - prior_union)
    branch_map = {name: branch_map[name] for name in REQUIRED_BRANCH_GROUPS}
    branch_memberships = [feature for values in branch_map.values() for feature in values]
    if len(branch_memberships) != len(set(branch_memberships)) or set(branch_memberships) != set(feature_names):
        raise RuntimeError(f"{condition_name} branch groups must be non-overlapping and exhaustive.")

    family_to_features = {}
    for feature in feature_names:
        family = None
        for prefix in ("qmatch__", "uaff__", "ucountlog__", "useen__", "item_label_n__"):
            if feature.startswith(prefix):
                family = feature.split("__", 1)[1]
                break
        if family is None:
            match = re.match(r"^user_(.+)_(?:recency_days|recent_count_180d)$", feature)
            family = match.group(1) if match else None
        if family and family != "brand":
            family_to_features.setdefault(family, []).append(feature)
    family_to_features = {
        family: sorted(set(features)) for family, features in sorted(family_to_features.items())
    }
    if not family_to_features:
        raise RuntimeError(f"{condition_name} functional facet-family mapping is empty.")
    return feature_names, mapping_df, semantic_map, branch_map, family_to_features


condition_feature_names = {}
condition_feature_mappings = {}
semantic_group_maps = {}
branch_group_maps = {}
facet_family_maps = {}
semantic_definition_rows = []
branch_definition_rows = []
facet_mapping_rows = []

semantic_definitions = {
    "retrieval_rank": "Stage-1 normalized retrieval score plus the learned candidate-rank embedding.",
    "history_depth_recency": "Leakage-safe history depth and user/facet recency features.",
    "functional_preference": "Functional candidate-profile affinity and profile concentration features.",
    "query_profile_interaction": "Direct executed query-profile interaction features only; no proxy substitution.",
    "candidate_history_interaction": "Candidate-specific prior count or prior-seen interaction features.",
    "brand_prior": "Candidate-brand affinity features from strictly pre-target history; candidate brand presence is excluded.",
}
branch_definitions = {
    "candidate_common": "Candidate-common query/item/retrieval evidence, including candidate-side brand presence when executed.",
    "functional_prior": "Non-brand functional history, preference, recency, and candidate-history evidence.",
    "brand_prior": "Strictly prior user-brand affinity evidence, distinct from functional facets.",
}

for condition_name, manifest in interpretation_manifests.items():
    feature_names, mapping_df, semantic_map, branch_map, family_map = feature_groups_for_condition(
        condition_name, manifest
    )
    condition_feature_names[condition_name] = feature_names
    condition_feature_mappings[condition_name] = mapping_df
    semantic_group_maps[condition_name] = semantic_map
    branch_group_maps[condition_name] = branch_map
    facet_family_maps[condition_name] = family_map
    for group in REQUIRED_SEMANTIC_GROUPS:
        features = semantic_map[group]
        semantic_definition_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "semantic_group": group,
            "definition": semantic_definitions[group],
            "available": bool(features),
            "feature_count": int(len(features)),
            "features_json": json.dumps(features, ensure_ascii=False),
            "rank_embedding_included_in_mask": bool(group == "retrieval_rank"),
            "unavailable_reason": (
                "No direct query-profile interaction feature exists in the executed contract; no proxy was invented."
                if group == "query_profile_interaction" and not features
                else (
                    "The P2-Q contract contains no user-prior features."
                    if condition_name == "P2-Q" and not features and group != "retrieval_rank"
                    else ""
                )
            ),
            "definition_selected_without_outcomes": True,
        })
    for group in REQUIRED_BRANCH_GROUPS:
        features = branch_map[group]
        branch_definition_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "branch_group": group,
            "definition": branch_definitions[group],
            "available": bool(features),
            "feature_count": int(len(features)),
            "features_json": json.dumps(features, ensure_ascii=False),
            "rank_embedding_included_in_mask": bool(group == "candidate_common"),
            "definition_selected_without_outcomes": True,
        })
    for family, features in family_map.items():
        for feature in features:
            facet_mapping_rows.append({
                "category": CATEGORY_LABEL,
                "category_id": CATEGORY_ID,
                "condition_name": condition_name,
                "facet_family": family,
                "feature": feature,
                "brand_excluded": True,
                "definition_selected_without_outcomes": True,
            })

semantic_group_definition_df = pd.DataFrame(semantic_definition_rows)
branch_group_definition_df = pd.DataFrame(branch_definition_rows)
facet_family_mapping_df = pd.DataFrame(facet_mapping_rows)


def load_primary_reranked(condition_name):
    path = reranked_paths[condition_name]
    frame = pd.read_parquet(path, filters=[("pool_depth", "==", PRIMARY_POOL_DEPTH)]).copy()
    require_unique_columns(frame, f"{condition_name} reranked candidates")
    required = [
        "case_id", "query_id", "user_id", "regime", "candidate_parent_asin",
        "candidate_item_id", "candidate_rank", "candidate_score",
        "candidate_score_norm_pool", "transformer_score", "pool_depth",
        "rerank_rank", "is_gt", "rerank_method",
    ]
    require_columns(frame, required, f"{condition_name} reranked candidates")
    expected_method = str(run_manifests[condition_name].get("rerank_method", "transformer_rerank"))
    frame = frame.loc[frame["rerank_method"].astype(str).eq(expected_method)].copy()
    if frame.empty:
        raise RuntimeError(f"No {condition_name} Transformer rows remain at depth {PRIMARY_POOL_DEPTH}.")
    for column in ["case_id", "query_id", "user_id", "candidate_parent_asin", "candidate_item_id"]:
        frame[column] = frame[column].astype(str)
    frame["regime"] = frame["regime"].astype(str).str.lower()
    frame["pool_depth"] = pd.to_numeric(frame["pool_depth"], errors="raise").astype(int)
    frame["label"] = pd.to_numeric(frame["is_gt"], errors="raise").astype(np.int8)
    if "candidate_source_slug" not in frame.columns:
        manifest = interpretation_manifests.get(condition_name, {})
        artifact_slugs = sorted({
            str(slug)
            for row in manifest.get("candidate_feature_oof_artifacts", [])
            if int(row.get("pool_depth", -1)) == PRIMARY_POOL_DEPTH
            for slug in row.get("candidate_source_slugs", [])
        })
        frame["candidate_source_slug"] = artifact_slugs[0] if len(artifact_slugs) == 1 else "single_source"

    frame["candidate_source_slug"] = frame["candidate_source_slug"].fillna("").astype(str).str.strip()

    frame["candidate_source_slug"] = frame["candidate_source_slug"].astype(str)
    if "prediction_source" not in frame.columns:
        if condition_name != "P2-Q":
            raise RuntimeError(f"{condition_name} reranked export lacks prediction_source.")
        frame["prediction_source"] = NO_PRIOR_NATIVE_SOURCE
    frame["prediction_source"] = frame["prediction_source"].fillna("").astype(str)
    unique_key = ["candidate_source_slug", "pool_depth", "case_id", "candidate_parent_asin"]
    if frame.duplicated(unique_key).any():
        raise RuntimeError(f"{condition_name} reranked candidates are duplicated at the candidate grain.")
    case_counts = frame.groupby("case_id", sort=False).size()
    if not case_counts.eq(PRIMARY_POOL_DEPTH).all():
        raise RuntimeError(f"{condition_name} does not contain exactly {PRIMARY_POOL_DEPTH} candidates per case.")
    if frame.groupby("case_id")["query_id"].nunique().gt(1).any():
        raise RuntimeError(f"{condition_name} case_id maps to multiple query_id values.")
    if frame.groupby("case_id")["user_id"].nunique().gt(1).any():
        raise RuntimeError(f"{condition_name} case_id maps to multiple user_id values.")
    if frame.groupby("case_id")["regime"].nunique().gt(1).any():
        raise RuntimeError(f"{condition_name} case_id maps to multiple regimes.")
    if frame.groupby("case_id")["prediction_source"].nunique().gt(1).any():
        raise RuntimeError(f"{condition_name} prediction_source varies within a case.")
    if frame.groupby("case_id")["label"].sum().gt(1).any():
        raise RuntimeError(f"{condition_name} contains more than one target row per case.")
    return frame.reset_index(drop=True)


reranked_primary = {
    condition_name: load_primary_reranked(condition_name)
    for condition_name in SOURCE_DIRS
}

case_identity = []
for condition_name, frame in reranked_primary.items():
    case_identity.append(
        frame[["case_id", "query_id", "user_id", "regime"]]
        .drop_duplicates("case_id")
        .assign(condition_name=condition_name)
    )
case_identity_df = pd.concat(case_identity, ignore_index=True)
for column in ["query_id", "user_id", "regime"]:
    if case_identity_df.groupby("case_id")[column].nunique().gt(1).any():
        raise RuntimeError(f"Cross-condition case identity mismatch in {column}.")


def primary_artifact_path(manifest):
    candidates = [
        row for row in manifest.get("candidate_feature_oof_artifacts", [])
        if int(row.get("pool_depth", -1)) == PRIMARY_POOL_DEPTH
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one held-out candidate artifact at depth {PRIMARY_POOL_DEPTH}, found {len(candidates)}."
        )
    return as_path(candidates[0].get("path"), "held-out candidate artifact")


heldout_reranked_prediction_qc_rows = []


def load_heldout_condition(condition_name, manifest):
    feature_names = condition_feature_names[condition_name]
    heldout_path = primary_artifact_path(manifest)
    if not heldout_path.exists():
        raise FileNotFoundError(f"Missing {condition_name} held-out candidate artifact: {heldout_path}")
    heldout = pd.read_parquet(heldout_path).copy()
    require_unique_columns(heldout, f"{condition_name} held-out artifact")
    required = [
        "condition_name", "category_id", "pool_depth", "case_id", "query_id",
        "candidate_parent_asin", "candidate_source_slug", "label", "fold_id",
        "oof_prediction", *feature_names,
    ]
    require_columns(heldout, required, f"{condition_name} held-out artifact")
    if set(heldout["condition_name"].astype(str)) != {condition_name}:
        raise RuntimeError(f"{condition_name} held-out condition label mismatch.")
    if set(heldout["category_id"].astype(str)) != {CATEGORY_ID}:
        raise RuntimeError(f"{condition_name} held-out category mismatch.")
    if set(pd.to_numeric(heldout["pool_depth"], errors="raise").astype(int)) != {PRIMARY_POOL_DEPTH}:
        raise RuntimeError(f"{condition_name} held-out depth mismatch.")
    for column in ["case_id", "query_id", "candidate_parent_asin", "candidate_source_slug"]:
        heldout[column] = heldout[column].fillna("").astype(str).str.strip()
    heldout["fold_id"] = pd.to_numeric(heldout["fold_id"], errors="raise").astype(int)
    heldout["label"] = pd.to_numeric(heldout["label"], errors="raise").astype(np.int8)
    heldout["oof_prediction"] = pd.to_numeric(heldout["oof_prediction"], errors="raise").astype(np.float32)
    for feature in feature_names:
        heldout[feature] = pd.to_numeric(heldout[feature], errors="raise").astype(np.float32)
    key = ["candidate_source_slug", "pool_depth", "query_id", "candidate_parent_asin"]
    if heldout.duplicated(key).any():
        raise RuntimeError(f"{condition_name} held-out artifact is duplicated at the candidate grain.")

    reranked = reranked_primary[condition_name].copy()
    reranked_columns = [
        "candidate_source_slug", "pool_depth", "case_id", "query_id", "user_id", "regime",
        "candidate_parent_asin", "candidate_item_id", "candidate_rank", "candidate_score",
        "candidate_score_norm_pool", "transformer_score", "prediction_source", "rerank_rank", "label",
    ]
    overlapping_value_columns = [
        column for column in reranked_columns
        if column in heldout.columns and column not in set(key) | {"case_id", "label"}
    ]
    reranked_rename = {
        "case_id": "reranked_case_id",
        "label": "reranked_label",
        **{column: f"reranked__{column}" for column in overlapping_value_columns},
    }
    reranked_join = reranked[reranked_columns].rename(columns=reranked_rename)
    if reranked_join.duplicated(key).any():
        raise RuntimeError(f"{condition_name} reranked join keys are duplicated.")
    before = len(heldout)
    merged = heldout.merge(reranked_join, on=key, how="left", validate="one_to_one", indicator=True)
    if len(merged) != before or not merged["_merge"].eq("both").all():
        raise RuntimeError(f"{condition_name} held-out/reranked join lost or multiplied rows.")
    merged = merged.drop(columns="_merge")
    if not merged["case_id"].eq(merged["reranked_case_id"]).all():
        raise RuntimeError(f"{condition_name} held-out/reranked case_id mismatch.")
    if not merged["label"].astype(int).eq(merged["reranked_label"].astype(int)).all():
        raise RuntimeError(f"{condition_name} held-out/reranked label mismatch.")
    for column in overlapping_value_columns:
        left = pd.to_numeric(merged[column], errors="coerce").to_numpy(dtype=float)
        right = pd.to_numeric(merged[f"reranked__{column}"], errors="coerce").to_numpy(dtype=float)
        if not np.allclose(left, right, rtol=0.0, atol=FALLBACK_IDENTITY_ATOL, equal_nan=True):
            raise RuntimeError(f"{condition_name} held-out/reranked {column} mismatch.")
    merged["reranked_transformer_score"] = pd.to_numeric(
        merged["transformer_score"], errors="raise"
    ).astype(np.float32)
    merged["reranked_rerank_rank"] = pd.to_numeric(
        merged["rerank_rank"], errors="raise"
    ).astype(np.int32)
    export_error = np.abs(
        merged["oof_prediction"].to_numpy(dtype=float)
        - merged["reranked_transformer_score"].to_numpy(dtype=float)
    )
    max_export_error = float(export_error.max()) if export_error.size else 0.0
    mismatch_count = int((export_error > FALLBACK_IDENTITY_ATOL).sum())
    merged["transformer_score"] = merged["oof_prediction"].astype(np.float32)
    recomputed_rerank_rank = rank_scores(merged, merged["transformer_score"])
    rerank_rank_mismatch_count = int(
        (merged["reranked_rerank_rank"].to_numpy(dtype=np.int32) != recomputed_rerank_rank).sum()
    )
    merged["rerank_rank"] = recomputed_rerank_rank.astype(np.int32)
    heldout_reranked_prediction_qc_rows.append({
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "condition_name": condition_name,
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "row_count": int(len(merged)),
        "mismatch_count": mismatch_count,
        "rerank_rank_mismatch_count": rerank_rank_mismatch_count,
        "maximum_absolute_error": max_export_error,
        "atol": FALLBACK_IDENTITY_ATOL,
        "passed": bool(mismatch_count == 0 and rerank_rank_mismatch_count == 0),
        "score_policy": "oof_prediction_authoritative_transformer_score_overwritten_for_interpretation",
        "rank_policy": "rerank_rank_recomputed_from_authoritative_oof_prediction",
    })
    if mismatch_count or rerank_rank_mismatch_count:
        print(
            f"Warning: {condition_name} held-out oof_prediction and reranked transformer_score/rank disagree "
            f"for score_rows={mismatch_count}, rank_rows={rerank_rank_mismatch_count}; "
            f"max_abs_error={max_export_error}. Notebook 19 uses oof_prediction and recomputed ranks "
            "as the authoritative held-out scoring surface."
        )
    merged = merged.drop(columns=[
        "reranked_case_id", "reranked_label",
        *[f"reranked__{column}" for column in overlapping_value_columns],
    ])
    allowed_sources = set(native_prediction_sources_for_condition(condition_name))
    if condition_name != "P2-Q":
        allowed_sources.add(FALLBACK_SOURCE)
    unexpected = sorted(set(merged["prediction_source"].astype(str)) - allowed_sources)
    if unexpected:
        raise RuntimeError(f"{condition_name} has unexpected prediction sources: {unexpected}")
    if condition_name == "P2-P":
        cold = merged["regime"].eq("cold")
        fallback = merged["prediction_source"].eq(FALLBACK_SOURCE)
        if not cold.eq(fallback).all():
            raise RuntimeError(f"{condition_name} strict-cold and fallback source flags disagree.")
    elif condition_name == "Full":
        cold = merged["regime"].eq("cold")
        fallback = merged["prediction_source"].eq(FALLBACK_SOURCE)
        if fallback.any() and not cold.eq(fallback).all():
            raise RuntimeError(f"{condition_name} strict-cold and fallback source flags disagree.")
        if cold.any() and not fallback.any() and merged["prediction_source"].eq(FULL_NATIVE_SOURCE).any():
            print(
                "Warning: Full uses native_full_self_pool_model without a separate p2q_cold_fallback "
                "source marker; treating those rows as native Full held-out scores."
            )
    return merged.reset_index(drop=True)


heldout_primary = {
    condition_name: load_heldout_condition(condition_name, manifest)
    for condition_name, manifest in interpretation_manifests.items()
}
heldout_reranked_prediction_qc_df = pd.DataFrame(heldout_reranked_prediction_qc_rows)
if not heldout_reranked_prediction_qc_df.empty and not heldout_reranked_prediction_qc_df["passed"].all():
    try:
        display(heldout_reranked_prediction_qc_df)
    except NameError:
        print(heldout_reranked_prediction_qc_df.to_string(index=False))

# Downstream diagnostic cells read reranked_primary; keep it aligned with the
# authoritative held-out scoring surface after the OOF score/rank reconciliation.
for _condition_name, _heldout_frame in heldout_primary.items():
    reranked_primary[_condition_name] = _heldout_frame.copy()

print("Interpretation conditions:", sorted(heldout_primary))
print("Primary held-out rows:", {key: len(value) for key, value in heldout_primary.items()})


,category,category_id,condition_name,candidate_pool_depth,row_count,mismatch_count,rerank_rank_mismatch_count,maximum_absolute_error,atol,passed,score_policy,rank_policy
0,Herbal Supplements,herbal,P2-Q,1000,1968000,0,0,0.000000,1.000000e-07,True,oof_prediction_authoritative_transformer_score...,rerank_rank_recomputed_from_authoritative_oof_...
1,Herbal Supplements,herbal,Full,1000,1968000,1967990,354203,0.059238,1.000000e-07,False,oof_prediction_authoritative_transformer_score...,rerank_rank_recomputed_from_authoritative_oof_...


Interpretation conditions: ['Full', 'P2-Q']
Primary held-out rows: {'P2-Q': 1968000, 'Full': 1968000}


In [6]:
# ==== Transformer Model Definition (Checkpoint-compatible) ====
class CandidateTransformerRanker(nn.Module):

    def __init__(
        self,
        n_features,
        d_model,
        nhead,
        num_layers,
        dim_feedforward,
        dropout,
        max_rank,
    ):
        super().__init__()
        self.input_norm = nn.LayerNorm(n_features)
        self.feature_proj = nn.Sequential(
            nn.Linear(n_features, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )
        self.rank_embed = nn.Embedding(max_rank, d_model, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.score_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, features, rank_pos, padding_mask):
        x = self.input_norm(features)
        x = self.feature_proj(x) + self.rank_embed(
            rank_pos.clamp(min=0, max=self.rank_embed.num_embeddings - 1)
        )
        x = self.encoder(x, src_key_padding_mask=padding_mask)
        scores = self.score_head(x).squeeze(-1)
        mask_value = -1e4 if scores.dtype == torch.float16 else -1e9
        return scores.masked_fill(padding_mask, mask_value)


def contract_for_primary_depth(manifest):
    rows = [
        row
        for row in manifest.get("model_contracts", [])
        if int(row.get("pool_depth", -1)) == PRIMARY_POOL_DEPTH
    ]
    if len(rows) != 1:
        raise RuntimeError(
            f"Expected one model contract at depth {PRIMARY_POOL_DEPTH}, found {len(rows)}."
        )
    path = as_path(rows[0].get("path"), "Transformer model contract")
    contract = read_json(path, "Transformer model contract")
    if int(contract.get("pool_depth", -1)) != PRIMARY_POOL_DEPTH:
        raise RuntimeError("Transformer model-contract depth mismatch.")
    return path, contract


condition_contracts = {}
condition_contract_paths = {}
condition_fold_contracts = {}
fold_lineage_rows = []

for condition_name, manifest in interpretation_manifests.items():
    contract_path, contract = contract_for_primary_depth(manifest)
    feature_names = condition_feature_names[condition_name]
    if contract.get("feature_columns") != feature_names:
        raise RuntimeError(
            f"{condition_name} checkpoint feature order differs from the interpretation contract."
        )
    if contract.get("settings") != manifest.get("model_configuration"):
        raise RuntimeError(
            f"{condition_name} checkpoint settings differ from the executed interpretation contract."
        )
    if contract.get("feature_dtypes") != manifest.get(
        "structured_feature_dtypes"
    ):
        raise RuntimeError(
            f"{condition_name} checkpoint feature dtypes differ from the interpretation contract."
        )
    condition_contracts[condition_name] = contract
    condition_contract_paths[condition_name] = contract_path
    condition_fold_contracts[condition_name] = {
        int(row["fold"]): row for row in contract.get("folds", [])
    }
    assignment_path = as_path(
        contract.get("fold_assignment_path"), "fold assignment artifact"
    )
    if not assignment_path.exists():
        raise FileNotFoundError(
            f"Missing {condition_name} fold assignment artifact: {assignment_path}"
        )
    assignment_df = pd.read_parquet(assignment_path).copy()
    require_columns(
        assignment_df,
        ["query_id", "case_id", "user_id", "fold"],
        f"{condition_name} fold assignments",
    )
    for column in ["query_id", "case_id", "user_id"]:
        assignment_df[column] = assignment_df[column].astype(str)
    assignment_df["fold"] = pd.to_numeric(
        assignment_df["fold"], errors="raise"
    ).astype(int)
    if assignment_df["query_id"].duplicated().any():
        raise RuntimeError(
            f"{condition_name} fold assignments duplicate query_id."
        )
    if assignment_df.groupby("user_id")["fold"].nunique().gt(1).any():
        raise RuntimeError(
            f"{condition_name} user-group-disjoint OOF lineage failed."
        )
    heldout_for_lineage = heldout_primary[condition_name]
    heldout_for_lineage = heldout_for_lineage.loc[
        heldout_for_lineage["prediction_source"].isin(
            native_prediction_sources_for_condition(condition_name)
        )
    ]
    observed = heldout_for_lineage[["query_id", "fold_id"]].drop_duplicates()
    if observed["query_id"].duplicated().any():
        raise RuntimeError(
            f"{condition_name} held-out query_id maps to multiple folds."
        )
    compared = observed.merge(
        assignment_df[["query_id", "fold"]],
        on="query_id",
        how="left",
        validate="one_to_one",
    )
    fold_match = bool(
        compared["fold"].notna().all()
        and compared["fold_id"]
        .astype(int)
        .eq(compared["fold"].astype(int))
        .all()
    )
    if not fold_match:
        raise RuntimeError(
            f"{condition_name} held-out fold IDs differ from the checkpoint assignment artifact."
        )
    fold_lineage_rows.append({
        "condition_name": condition_name,
        "assignment_path": str(assignment_path),
        "query_count_checked": int(len(compared)),
        "user_count_checked": int(assignment_df["user_id"].nunique()),
        "user_group_disjoint": True,
        "heldout_fold_assignment_exact": True,
    })

_full_model_role = str(run_manifests.get("Full", {}).get("shared_all_prior_model_role", "train_and_export"))
if _full_model_role == "load_and_score":
    if "P2-P" in condition_contract_paths and "Full" in condition_contract_paths:
        if condition_contract_paths["P2-P"].resolve() != condition_contract_paths["Full"].resolve():
            raise RuntimeError("Full does not point to the exact RankP shared model contract.")
    if "P2-P" in condition_feature_names and "Full" in condition_feature_names:
        if condition_feature_names["P2-P"] != condition_feature_names["Full"]:
            raise RuntimeError("RankP and Full structured feature schemas differ for load-and-score lineage.")
elif _full_model_role != "train_and_export":
    raise RuntimeError(f"Unsupported Full shared_all_prior_model_role: {_full_model_role}")


def load_fold_model(condition_name, fold_id):
    fold_info = condition_fold_contracts[condition_name].get(int(fold_id))
    if fold_info is None:
        raise RuntimeError(
            f"{condition_name} model contract lacks fold {fold_id}."
        )
    if bool(fold_info.get("fallback_used", False)):
        return None, None, fold_info
    model_path = as_path(
        fold_info.get("model_path"), f"{condition_name} fold checkpoint"
    )
    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing {condition_name} fold checkpoint: {model_path}"
        )
    payload = safe_torch_load(model_path)
    feature_names = condition_feature_names[condition_name]
    if payload.get("feature_columns") != feature_names:
        raise RuntimeError(
            f"{condition_name} fold {fold_id} checkpoint feature mismatch."
        )
    settings = payload.get("settings", payload.get("model_configuration", {}))
    required_settings = [
        "d_model",
        "nhead",
        "num_layers",
        "dim_feedforward",
        "dropout",
        "max_rank",
    ]
    missing_settings = [
        name for name in required_settings if name not in settings
    ]
    if missing_settings:
        raise RuntimeError(
            f"{condition_name} fold {fold_id} checkpoint settings missing: {missing_settings}"
        )
    model = CandidateTransformerRanker(
        n_features=len(feature_names),
        d_model=int(settings["d_model"]),
        nhead=int(settings["nhead"]),
        num_layers=int(settings["num_layers"]),
        dim_feedforward=int(settings["dim_feedforward"]),
        dropout=float(settings["dropout"]),
        max_rank=int(settings["max_rank"]),
    )
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return model, payload, fold_info


@torch.no_grad()
def score_fold_frame(
    model, payload, frame, mask_features=None, mask_rank_embedding=False
):
    feature_names = list(payload["feature_columns"])
    mean = payload["mean"].detach().cpu().numpy().astype(np.float32)
    std = payload["std"].detach().cpu().numpy().astype(np.float32)
    if mean.shape != (len(feature_names),) or std.shape != (
        len(feature_names),
    ):
        raise RuntimeError(
            "Checkpoint standardizer shape differs from the feature schema."
        )
    mask_features = list(mask_features or [])
    missing_mask_features = sorted(set(mask_features) - set(feature_names))
    if missing_mask_features:
        raise RuntimeError(
            f"Mask target contains unknown features: {missing_mask_features}"
        )
    mask_positions = [feature_names.index(feature) for feature in mask_features]

    ordered = frame.copy()
    ordered["_input_row"] = np.arange(len(ordered), dtype=np.int64)
    ordered = ordered.sort_values(
        ["query_id", "candidate_rank", "candidate_parent_asin"],
        kind="mergesort",
    )
    examples = []
    for query_id, group in ordered.groupby("query_id", sort=True):
        values = (
            group[feature_names]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .to_numpy(dtype=np.float32, copy=True)
        )
        values = ((values - mean) / std).astype(np.float32, copy=False)
        if mask_positions:
            values[:, mask_positions] = 0.0
        if mask_rank_embedding:
            rank_pos = np.zeros(len(group), dtype=np.int64)
        else:
            rank_pos = (
                pd.to_numeric(group["candidate_rank"], errors="raise")
                .astype(int)
                .clip(lower=1, upper=int(model.rank_embed.num_embeddings) - 1)
                .to_numpy(dtype=np.int64)
            )
        examples.append({
            "query_id": str(query_id),
            "row_ids": group["_input_row"].to_numpy(dtype=np.int64),
            "features": values,
            "rank_pos": rank_pos,
            "length": int(len(group)),
        })

    settings = payload.get("settings", {})
    batch_policy = settings.get("batch_size_by_depth", {})
    batch_size = int(
        batch_policy.get(
            str(PRIMARY_POOL_DEPTH), batch_policy.get(PRIMARY_POOL_DEPTH, 6)
        )
    )
    predictions = np.full(len(frame), np.nan, dtype=np.float32)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        max_len = max(example["length"] for example in batch)
        n_features = len(feature_names)
        features = torch.zeros(
            len(batch), max_len, n_features, dtype=torch.float32, device=device
        )
        rank_pos = torch.zeros(
            len(batch), max_len, dtype=torch.long, device=device
        )
        padding_mask = torch.ones(
            len(batch), max_len, dtype=torch.bool, device=device
        )
        for batch_index, example in enumerate(batch):
            length = example["length"]
            features[batch_index, :length] = torch.from_numpy(
                example["features"]
            ).to(device)
            rank_pos[batch_index, :length] = torch.from_numpy(
                example["rank_pos"]
            ).to(device)
            padding_mask[batch_index, :length] = False
        scores = model(features, rank_pos, padding_mask).detach().cpu().numpy()
        for batch_index, example in enumerate(batch):
            length = example["length"]
            predictions[example["row_ids"]] = scores[
                batch_index, :length
            ].astype(np.float32)
    model = model.to("cpu")
    if not np.isfinite(predictions).all():
        raise RuntimeError(
            "Transformer scoring produced missing or non-finite predictions."
        )
    return predictions


reproduction_rows = []
selected_checkpoint_performance_rows = []
native_primary = {}

for condition_name, heldout in heldout_primary.items():
    native_sources = native_prediction_sources_for_condition(condition_name)
    native_source = observed_native_prediction_source_label(condition_name, heldout)
    candidate_native = heldout["prediction_source"].isin(native_sources)

    fallback_excluded = int(heldout["prediction_source"].eq(FALLBACK_SOURCE).sum())
    native_parts = []

    for fold_id, fold_df in heldout.loc[candidate_native].groupby(
        "fold_id", sort=True
    ):
        fold_info = condition_fold_contracts[condition_name].get(
            int(fold_id), {}
        )
        model_fallback = bool(fold_info.get("fallback_used", False))

        if model_fallback:
            reproduction_rows.append({
                "category": CATEGORY_LABEL,
                "category_id": CATEGORY_ID,
                "condition_name": condition_name,
                "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                "fold_id": int(fold_id),
                "native_prediction_source": native_source,
                "excluded_prediction_source": FALLBACK_SOURCE,
                "native_transformer_row_count": 0,
                "cold_fallback_row_count_excluded": int(
                    heldout.loc[
                        heldout["fold_id"].eq(int(fold_id))
                        & heldout["prediction_source"].eq(FALLBACK_SOURCE)
                    ].shape[0]
                ),
                "model_level_fallback_row_count_excluded": int(len(fold_df)),
                "maximum_absolute_native_export_error": np.nan,
                "maximum_absolute_ndcg_at_5_error": np.nan,
                "target_rank_mismatch_count": 0,
                "top5_status_mismatch_count": 0,
                "reproduction_atol": REPRODUCTION_ATOL,
                "prediction_reproduced": True,
                "ndcg_at_5_reproduced": True,
                "top5_status_reproduced": True,
                "interpretation_scoring_available": True,
                "details": "Model-level fallback fold excluded from native Transformer reproduction.",
            })
            continue

        model, payload, _ = load_fold_model(condition_name, int(fold_id))
        reproduced = score_fold_frame(model, payload, fold_df)
        exported = fold_df["oof_prediction"].to_numpy(dtype=float)

        interpretation_scoring_available = bool(
            len(reproduced) == len(fold_df) and np.isfinite(reproduced).all()
        )
        maximum_error = float(
            np.max(np.abs(reproduced.astype(float) - exported))
        )

        exported_case_metrics = case_metric_frame(fold_df, exported).set_index(
            "case_id"
        )
        reproduced_case_metrics = case_metric_frame(
            fold_df, reproduced
        ).set_index("case_id")
        reproduced_case_metrics = reproduced_case_metrics.reindex(
            exported_case_metrics.index
        )

        target_rank_mismatch_count = int(
            (
                ~exported_case_metrics["target_rank"]
                .fillna(-1)
                .astype(int)
                .eq(
                    reproduced_case_metrics["target_rank"]
                    .fillna(-1)
                    .astype(int)
                )
            ).sum()
        )

        maximum_metric_error = float(
            np.max(
                np.abs(
                    exported_case_metrics["NDCG@5"].to_numpy(dtype=float)
                    - reproduced_case_metrics["NDCG@5"].to_numpy(dtype=float)
                )
            )
        )

        exported_top5 = (
            exported_case_metrics["target_rank"]
            .fillna(10**9)
            .astype(int)
            .le(PRIMARY_K)
        )
        reproduced_top5 = (
            reproduced_case_metrics["target_rank"]
            .fillna(10**9)
            .astype(int)
            .le(PRIMARY_K)
        )
        top5_status_mismatch_count = int((~exported_top5.eq(reproduced_top5)).sum())

        raw_prediction_reproduced = bool(maximum_error <= REPRODUCTION_ATOL)
        ndcg_at_5_reproduced = bool(maximum_metric_error <= 1e-12)
        top5_status_reproduced = bool(top5_status_mismatch_count == 0)

        reproduction_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "candidate_pool_depth": PRIMARY_POOL_DEPTH,
            "fold_id": int(fold_id),
            "native_prediction_source": native_source,
            "excluded_prediction_source": FALLBACK_SOURCE,
            "native_transformer_row_count": int(len(fold_df)),
            "cold_fallback_row_count_excluded": int(
                heldout.loc[
                    heldout["fold_id"].eq(int(fold_id))
                    & heldout["prediction_source"].eq(FALLBACK_SOURCE)
                ].shape[0]
            ),
            "model_level_fallback_row_count_excluded": 0,
            "maximum_absolute_native_export_error": maximum_error,
            "maximum_absolute_ndcg_at_5_error": maximum_metric_error,
            "target_rank_mismatch_count": target_rank_mismatch_count,
            "top5_status_mismatch_count": top5_status_mismatch_count,
            "reproduction_atol": REPRODUCTION_ATOL,
            "prediction_reproduced": raw_prediction_reproduced,
            "ndcg_at_5_reproduced": ndcg_at_5_reproduced,
            "top5_status_reproduced": top5_status_reproduced,
            "interpretation_scoring_available": interpretation_scoring_available,
            "details": "Exported oof_prediction is the authoritative held-out baseline; reloaded checkpoint scores are audited and used for masking counterfactuals.",
        })

        if not interpretation_scoring_available:
            raise RuntimeError(
                f"{condition_name} fold {fold_id} checkpoint could not score native held-out rows for masking: "
                f"row_count={len(fold_df)}, raw_score_error={maximum_error}"
            )

        kept = fold_df.copy()
        kept["authoritative_oof_prediction"] = exported
        kept["reproduced_prediction"] = reproduced
        native_parts.append(kept)

        case_metrics = case_metric_frame(fold_df, exported)
        for user_scope, regime, scope_df in metric_scopes(case_metrics):
            selected_checkpoint_performance_rows.append({
                "category": CATEGORY_LABEL,
                "category_id": CATEGORY_ID,
                "condition_name": condition_name,
                "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                "fold_id": int(fold_id),
                "user_scope": user_scope,
                "regime": regime,
                "case_count": int(len(scope_df)),
                "selected_checkpoint_heldout_ndcg_at_5": (
                    float(scope_df["NDCG@5"].mean()) if len(scope_df) else np.nan
                ),
            })

        del model, payload
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not native_parts:
        raise RuntimeError(
            f"No native Transformer fold rows remain for {condition_name}."
        )

    native_primary[condition_name] = pd.concat(native_parts, ignore_index=True)

    if (
        condition_name != "P2-Q"
        and fallback_excluded
        and not heldout.loc[
            heldout["prediction_source"].eq(FALLBACK_SOURCE), "regime"
        ]
        .eq("cold")
        .all()
    ):
        raise RuntimeError(
            f"{condition_name} fallback exclusion contains non-cold rows."
        )

reproduction_qc_df = pd.DataFrame(reproduction_rows)
selected_checkpoint_performance_df = pd.DataFrame(
    selected_checkpoint_performance_rows
)

scored_reproduction_qc = reproduction_qc_df.loc[
    pd.to_numeric(
        reproduction_qc_df["native_transformer_row_count"], errors="coerce"
    ).gt(0)
].copy()

if not scored_reproduction_qc["interpretation_scoring_available"].all():
    raise RuntimeError(
        "At least one native Transformer fold could not be scored for masking."
    )

reproduction_drift = scored_reproduction_qc.loc[
    ~scored_reproduction_qc["prediction_reproduced"]
    | ~scored_reproduction_qc["ndcg_at_5_reproduced"]
    | ~scored_reproduction_qc["top5_status_reproduced"]
].copy()

if not reproduction_drift.empty:
    print(
        "Warning: reloaded Transformer checkpoint scores do not exactly reproduce exported OOF scores. "
        "Exported oof_prediction remains authoritative; reloaded checkpoints are used only for masking."
    )
    drift_columns = [
        "condition_name",
        "fold_id",
        "native_transformer_row_count",
        "maximum_absolute_native_export_error",
        "maximum_absolute_ndcg_at_5_error",
        "target_rank_mismatch_count",
        "top5_status_mismatch_count",
    ]
    display(reproduction_drift[drift_columns])

print(reproduction_qc_df)

,condition_name,fold_id,native_transformer_row_count,maximum_absolute_native_export_error,maximum_absolute_ndcg_at_5_error,target_rank_mismatch_count,top5_status_mismatch_count
0,P2-Q,1,394000,0.002478,0.0,3,0
1,P2-Q,2,394000,0.003516,0.0,5,0
2,P2-Q,3,394000,0.007749,0.0,8,0
3,P2-Q,4,393000,0.004969,0.0,8,0
4,P2-Q,5,393000,0.003741,0.0,12,0
5,Full,1,265000,0.004490,0.0,6,0
6,Full,2,252000,0.003625,0.0,4,0
7,Full,3,269000,0.004746,0.0,6,0
8,Full,4,268000,0.006142,0.0,11,0
9,Full,5,258000,0.006542,0.0,3,0


             category category_id condition_name  candidate_pool_depth  \
0  Herbal Supplements      herbal           P2-Q                  1000   
1  Herbal Supplements      herbal           P2-Q                  1000   
2  Herbal Supplements      herbal           P2-Q                  1000   
3  Herbal Supplements      herbal           P2-Q                  1000   
4  Herbal Supplements      herbal           P2-Q                  1000   
5  Herbal Supplements      herbal           Full                  1000   
6  Herbal Supplements      herbal           Full                  1000   
7  Herbal Supplements      herbal           Full                  1000   
8  Herbal Supplements      herbal           Full                  1000   
9  Herbal Supplements      herbal           Full                  1000   

   fold_id     native_prediction_source excluded_prediction_source  \
0        1              no_prior_native          p2q_cold_fallback   
1        2              no_prior_native      

In [7]:
# ==== Held-out Semantic-Group Masking Diagnostics ====
masking_rows = []

for condition_name, native_df in native_primary.items():
    semantic_targets = [
        ("semantic_group", name, features, name in {"retrieval_rank"})
        for name, features in semantic_group_maps[condition_name].items()
        if features or name == "retrieval_rank"
    ]
    branch_targets = [
        ("branch", name, features, name == "candidate_common")
        for name, features in branch_group_maps[condition_name].items()
        if features
    ]
    facet_targets = [
        ("facet_family", family, features, False)
        for family, features in facet_family_maps[condition_name].items()
        if features
    ]
    targets = semantic_targets + branch_targets + facet_targets

    for fold_id, fold_df in native_df.groupby("fold_id", sort=True):
        model, payload, _ = load_fold_model(condition_name, int(fold_id))
        authoritative_scores = pd.to_numeric(
            fold_df.get("authoritative_oof_prediction", fold_df["oof_prediction"]),
            errors="raise",
        ).to_numpy(dtype=float)

        baseline_case_metrics = case_metric_frame(fold_df, authoritative_scores)
        baseline_scope_map = {
            (user_scope, regime): scope_df
            for user_scope, regime, scope_df in metric_scopes(baseline_case_metrics)
        }

        for target_type, target_name, target_features, mask_rank_embedding in targets:
            masked_scores = score_fold_frame(
                model,
                payload,
                fold_df,
                mask_features=target_features,
                mask_rank_embedding=mask_rank_embedding,
            )
            masked_case_metrics = case_metric_frame(fold_df, masked_scores).set_index("case_id")
            for (user_scope, regime), baseline_scope in baseline_scope_map.items():
                case_ids = baseline_scope["case_id"].astype(str).tolist()
                if not case_ids:
                    continue
                baseline_value = float(baseline_scope["NDCG@5"].mean())
                masked_value = float(masked_case_metrics.loc[case_ids, "NDCG@5"].mean())
                masking_rows.append({
                    "category": CATEGORY_LABEL,
                    "category_id": CATEGORY_ID,
                    "condition_name": condition_name,
                    "model_family": MODEL_FAMILY,
                    "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                    "fold_id": int(fold_id),
                    "user_scope": user_scope,
                    "regime": regime,
                    "target_type": target_type,
                    "target_name": target_name,
                    "target_feature_count": int(len(target_features)),
                    "target_features_json": json.dumps(target_features, ensure_ascii=False),
                    "rank_embedding_masked": bool(mask_rank_embedding),
                    "case_count": int(len(case_ids)),
                    "baseline_ndcg_at_5": baseline_value,
                    "masked_ndcg_at_5": masked_value,
                    "importance_ndcg_at_5_decrease": baseline_value - masked_value,
                    "mask_reference": "training_fold_mean_standardized_zero",
                    "native_transformer_only": True,
                    "heldout_only": True,
                    "causal_claim": False,
                })
        del model, payload
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

masking_by_fold_df = pd.DataFrame(masking_rows)
if masking_by_fold_df.empty:
    raise RuntimeError("No held-out Transformer masking results were produced.")
masking_summary_df = summarize_masking(masking_by_fold_df)
semantic_group_stability_df = build_masking_stability(masking_by_fold_df, "semantic_group")
semantic_group_fold_rank_correlation_df = build_fold_rank_correlation(
    masking_by_fold_df, "semantic_group"
)

semantic_observed = masking_summary_df.loc[
    masking_summary_df["target_type"].eq("semantic_group")
    & (
        masking_summary_df["user_scope"].eq("overall")
        | (
            masking_summary_df["user_scope"].eq("regime")
            & masking_summary_df["regime"].eq("strong")
        )
    )
].copy().rename(columns={"target_name": "semantic_group"})
report_scopes_df = pd.DataFrame([
    {"user_scope": "overall", "regime": "overall"},
    {"user_scope": "regime", "regime": "strong"},
])
semantic_grid = semantic_group_definition_df.merge(report_scopes_df, how="cross")
primary_semantic_group_summary_df = semantic_grid.merge(
    semantic_observed,
    on=[
        "category", "category_id", "condition_name", "semantic_group",
        "user_scope", "regime",
    ],
    how="left",
    validate="one_to_one",
)
primary_semantic_group_summary_df["candidate_pool_depth"] = PRIMARY_POOL_DEPTH
primary_semantic_group_summary_df["target_type"] = "semantic_group"
primary_semantic_group_summary_df["importance_available"] = (
    primary_semantic_group_summary_df["available"]
    & primary_semantic_group_summary_df["importance_mean"].notna()
)
if primary_semantic_group_summary_df.loc[
    primary_semantic_group_summary_df["available"], "importance_mean"
].isna().any():
    raise RuntimeError("An available primary semantic group lacks masking results.")

facet_observed = masking_summary_df.loc[
    masking_summary_df["target_type"].eq("facet_family")
    & (
        masking_summary_df["user_scope"].eq("overall")
        | (
            masking_summary_df["user_scope"].eq("regime")
            & masking_summary_df["regime"].eq("strong")
        )
    )
].copy().rename(columns={"target_name": "facet_family"})
primary_facet_family_summary_df = facet_observed.copy()

branch_contribution_df = masking_summary_df.loc[
    masking_summary_df["target_type"].eq("branch")
].copy().rename(columns={"target_name": "branch_group"})
branch_contribution_df = branch_contribution_df.merge(
    branch_group_definition_df[[
        "category", "category_id", "condition_name", "branch_group", "definition", "available"
    ]],
    on=["category", "category_id", "condition_name", "branch_group"],
    how="left",
    validate="many_to_one",
)

cross_model_summary_df = semantic_observed[[
    "category", "category_id", "regime", "user_scope", "model_family",
    "condition_name", "candidate_pool_depth", "semantic_group", "fold_count",
    "importance_mean", "importance_std_across_folds", "importance_ci_95_low",
    "importance_ci_95_high",
]].copy().rename(columns={
    "semantic_group": "feature_group",
    "importance_mean": "group_importance",
    "importance_std_across_folds": "uncertainty_across_folds",
    "importance_ci_95_low": "group_importance_ci_95_low",
    "importance_ci_95_high": "group_importance_ci_95_high",
})
cross_model_summary_df["direction_value"] = np.nan
cross_model_summary_df["importance_method"] = "heldout_group_mean_masking_ndcg_at_5_decrease"
cross_model_summary_df["direction_method"] = "not_estimated"
cross_model_summary_df["primary_report_depth"] = True
cross_model_summary_df["native_transformer_only"] = True
cross_model_summary_df["heldout_only"] = True
cross_model_summary_df["causal_claim"] = False

print(primary_semantic_group_summary_df[[
    "condition_name", "semantic_group", "available", "importance_mean", "fold_count"
]])


   condition_name                 semantic_group  available  importance_mean  \
0            P2-Q                 retrieval_rank       True        -0.017339   
1            P2-Q                 retrieval_rank       True        -0.024736   
2            P2-Q          history_depth_recency      False              NaN   
3            P2-Q          history_depth_recency      False              NaN   
4            P2-Q          functional_preference      False              NaN   
5            P2-Q          functional_preference      False              NaN   
6            P2-Q      query_profile_interaction      False              NaN   
7            P2-Q      query_profile_interaction      False              NaN   
8            P2-Q  candidate_history_interaction      False              NaN   
9            P2-Q  candidate_history_interaction      False              NaN   
10           P2-Q                    brand_prior      False              NaN   
11           P2-Q                    bra

In [9]:
# ==== Case Diagnostics Builder ====
def build_case_diagnostics(condition_name, frame):
    rows = []
    for case_id, group in frame.groupby("case_id", sort=False):
        group = group.copy()
        positives = group.loc[group["label"].astype(int).eq(1)]
        negatives = group.loc[group["label"].astype(int).eq(0)]
        target_present = len(positives) == 1
        target_score = float(positives["transformer_score"].iloc[0]) if target_present else np.nan
        mean_negative_score = float(negatives["transformer_score"].mean()) if len(negatives) else np.nan
        top_negative_score = float(negatives["transformer_score"].max()) if len(negatives) else np.nan
        original_target_rank = int(positives["candidate_rank"].iloc[0]) if target_present else None
        target_rank = int(positives["rerank_rank"].iloc[0]) if target_present else None
        if not target_present:
            taxonomy = "target_absent_from_pool"
        elif original_target_rank > PRIMARY_K and target_rank <= PRIMARY_K:
            taxonomy = "top5_entry"
        elif original_target_rank <= PRIMARY_K and target_rank > PRIMARY_K:
            taxonomy = "top5_loss"
        elif original_target_rank <= PRIMARY_K and target_rank <= PRIMARY_K:
            taxonomy = "top5_retained"
        else:
            taxonomy = "remains_below_top5"
        ndcg5 = (
            float(1.0 / np.log2(target_rank + 1.0))
            if target_rank is not None and target_rank <= PRIMARY_K
            else 0.0
        )
        rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "candidate_pool_depth": PRIMARY_POOL_DEPTH,
            "case_id": str(case_id),
            "query_id": str(group["query_id"].iloc[0]),
            "user_id": str(group["user_id"].iloc[0]),
            "regime": str(group["regime"].iloc[0]),
            "prediction_source": str(group["prediction_source"].iloc[0]),
            "target_present": bool(target_present),
            "original_target_rank": original_target_rank,
            "target_rank": target_rank,
            "target_score": target_score,
            "mean_negative_score": mean_negative_score,
            "top_negative_score": top_negative_score,
            "target_score_margin": target_score - mean_negative_score if target_present else np.nan,
            "target_score_margin_reference": "mean_negative_candidate_score",
            "target_vs_top_negative_margin": target_score - top_negative_score if target_present else np.nan,
            "top5_taxonomy": taxonomy,
            "diagnostic_NDCG@5_from_reranked_rank": ndcg5,
            "diagnostic_HitRate@5_from_reranked_rank": float(target_rank is not None and target_rank <= PRIMARY_K),
        })
    return pd.DataFrame(rows)


margin_diagnostics_df = pd.concat([
    build_case_diagnostics(condition_name, frame)
    for condition_name, frame in reranked_primary.items()
], ignore_index=True)

taxonomy_rows = []
for condition_name, condition_df in margin_diagnostics_df.groupby("condition_name", sort=True):
    scopes = [
        ("all_scored_rows", condition_df),
        (
            "native_transformer_only",
            condition_df.loc[~condition_df["prediction_source"].eq(FALLBACK_SOURCE)],
        ),
    ]
    for prediction_scope, scope_df in scopes:
        report_scopes = [("overall", "overall", scope_df)]
        report_scopes.extend([
            ("regime", regime, scope_df.loc[scope_df["regime"].eq(regime)])
            for regime in sorted(scope_df["regime"].dropna().astype(str).unique())
        ])
        for user_scope, regime, report_df in report_scopes:
            if report_df.empty:
                continue
            for taxonomy, taxonomy_df in report_df.groupby("top5_taxonomy", sort=True):
                taxonomy_rows.append({
                    "category": CATEGORY_LABEL,
                    "category_id": CATEGORY_ID,
                    "condition_name": condition_name,
                    "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                    "prediction_scope": prediction_scope,
                    "user_scope": user_scope,
                    "regime": regime,
                    "top5_taxonomy": taxonomy,
                    "case_count": int(len(taxonomy_df)),
                    "case_rate": float(len(taxonomy_df) / len(report_df)),
                    "evidence_role": "rank_movement_diagnostic_not_canonical_performance",
                })

top5_taxonomy_df = pd.DataFrame(taxonomy_rows)

_prediction_scope_map = (
    margin_diagnostics_df[
        ["condition_name", "case_id", "prediction_source", "target_present",
         "target_score_margin", "target_vs_top_negative_margin"]
    ]
    .drop_duplicates(["condition_name", "case_id"])
)
canonical_case_metrics_df = notebook14_canonical_metrics_df.merge(
    _prediction_scope_map,
    on=["condition_name", "case_id"],
    how="left",
    validate="one_to_one",
)
if canonical_case_metrics_df[["prediction_source"]].isna().any().any():
    raise RuntimeError("Notebook 14 canonical cases do not align with Transformer reranked outputs.")

for _condition_name in sorted(reranked_primary):
    _expected_cases = set(reranked_primary[_condition_name]["case_id"].astype(str))
    _canonical_cases = set(
        canonical_case_metrics_df.loc[
            canonical_case_metrics_df["condition_name"].eq(_condition_name), "case_id"
        ].astype(str)
    )
    if _expected_cases != _canonical_cases:
        raise RuntimeError(
            f"Notebook 14 canonical case universe differs for {_condition_name}: "
            f"expected={len(_expected_cases)}, canonical={len(_canonical_cases)}"
        )

regime_rows = []
for condition_name, condition_df in canonical_case_metrics_df.groupby("condition_name", sort=True):
    scopes = [
        ("all_scored_rows", condition_df),
        (
            "native_transformer_only",
            condition_df.loc[~condition_df["prediction_source"].eq(FALLBACK_SOURCE)],
        ),
    ]
    for prediction_scope, scope_df in scopes:
        if scope_df.empty:
            continue
        regime_means = (
            scope_df.groupby("regime", as_index=False, observed=True)
            .agg(
                regime_case_count=("case_id", "nunique"),
                NDCG_at_5=("NDCG@5", "mean"),
                HitRate_at_5=("HitRate@5", "mean"),
            )
        )
        diagnostic_overall = scope_df[
            ["target_present", "target_score_margin", "target_vs_top_negative_margin"]
        ]
        regime_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "candidate_pool_depth": PRIMARY_POOL_DEPTH,
            "prediction_scope": prediction_scope,
            "user_scope": "overall",
            "regime": "overall",
            "case_count": int(scope_df["case_id"].nunique()),
            "target_in_pool_rate": float(diagnostic_overall["target_present"].mean()),
            "NDCG@5": float(regime_means["NDCG_at_5"].mean()),
            "HitRate@5": float(regime_means["HitRate_at_5"].mean()),
            "target_score_margin_mean": float(diagnostic_overall["target_score_margin"].mean()),
            "target_vs_top_negative_margin_mean": float(
                diagnostic_overall["target_vs_top_negative_margin"].mean()
            ),
            "overall_weighting": "equal_regime_macro",
            "performance_authority": "Notebook14_pipeline_canonical_per_case_metrics",
            "margin_authority": "Notebook19_descriptive_diagnostic",
        })
        for regime, report_df in scope_df.groupby("regime", sort=True, observed=True):
            diagnostic_regime = report_df[
                ["target_present", "target_score_margin", "target_vs_top_negative_margin"]
            ]
            regime_rows.append({
                "category": CATEGORY_LABEL,
                "category_id": CATEGORY_ID,
                "condition_name": condition_name,
                "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                "prediction_scope": prediction_scope,
                "user_scope": "regime",
                "regime": str(regime),
                "case_count": int(report_df["case_id"].nunique()),
                "target_in_pool_rate": float(diagnostic_regime["target_present"].mean()),
                "NDCG@5": float(report_df["NDCG@5"].mean()),
                "HitRate@5": float(report_df["HitRate@5"].mean()),
                "target_score_margin_mean": float(diagnostic_regime["target_score_margin"].mean()),
                "target_vs_top_negative_margin_mean": float(
                    diagnostic_regime["target_vs_top_negative_margin"].mean()
                ),
                "overall_weighting": "within_regime_mean",
                "performance_authority": "Notebook14_pipeline_canonical_per_case_metrics",
                "margin_authority": "Notebook19_descriptive_diagnostic",
            })

regime_performance_df = pd.DataFrame(regime_rows)
for condition_name in reranked_primary:
    required_scopes = regime_performance_df.loc[
        regime_performance_df["condition_name"].eq(condition_name)
        & regime_performance_df["prediction_scope"].eq("all_scored_rows"),
        ["user_scope", "regime"],
    ]
    if not (
        ((required_scopes["user_scope"] == "overall") & (required_scopes["regime"] == "overall")).any()
        and ((required_scopes["user_scope"] == "regime") & (required_scopes["regime"] == "strong")).any()
    ):
        raise RuntimeError(f"{condition_name} must report canonical Overall and Strong performance at depth 1000.")


def s2q_oof_path_from_manifests():
    candidate_values = []

    for condition_name in ("P2-P", "Full", "P2-Q"):
        manifest = run_manifests[condition_name]

        for key in (
            "p2q_oof_candidate_predictions_path",
            "p2q_oof_prediction_path",
            "s2q_oof_candidate_predictions_path",
            "s2q_oof_prediction_path",
        ):
            value = manifest.get(key)
            if value:
                candidate_values.append(str(value))

        for key in (
            "p2q_oof_candidate_predictions",
            "p2q_oof_predictions",
            "s2q_oof_candidate_predictions",
            "s2q_oof_predictions",
        ):
            value = manifest.get("output_paths", {}).get(key)
            if value:
                candidate_values.append(str(value))

    candidate_values.append(str(S2Q_DIR / "p2q_oof_candidate_predictions.parquet"))

    existing = [Path(value) for value in dict.fromkeys(candidate_values) if
    Path(value).exists()]
    if not existing:
        raise FileNotFoundError(
            "No exact same-category P2-Q OOF artifact was found. Checked: "
            + json.dumps(candidate_values, ensure_ascii=False, indent=2)
        )

    names = {path.name for path in existing}
    allowed_names = {
        "p2q_oof_candidate_predictions.parquet",
        "s2q_oof_candidate_predictions.parquet",
    }
    if not names.issubset(allowed_names):
        raise RuntimeError(f"Unexpected Base OOF artifact names: {sorted(names)}")

    return existing[0]


s2q_oof_path = s2q_oof_path_from_manifests()
s2q_oof_df = pd.read_parquet(
    s2q_oof_path, filters=[("pool_depth", "==", PRIMARY_POOL_DEPTH)]
).copy()
require_columns(
    s2q_oof_df,
    [
        "query_id", "candidate_item_id", "pool_depth", "original_candidate_rank",
        "original_candidate_score", "rerank_score", "rerank_rank", "is_gt", "fold_id",
    ],
    "P2-Q OOF predictions",
)
s2q_oof_df["query_id"] = s2q_oof_df["query_id"].astype(str)
s2q_oof_df["candidate_item_id"] = s2q_oof_df["candidate_item_id"].astype(str)
if s2q_oof_df.duplicated(["query_id", "pool_depth", "candidate_item_id"]).any():
    raise RuntimeError("Base OOF predictions are duplicated at the candidate grain.")

cold_fallback_rows = []
for condition_name in ("P2-P", "Full"):
    fallback = reranked_primary[condition_name].loc[
        reranked_primary[condition_name]["prediction_source"].eq(FALLBACK_SOURCE)
    ].copy()
    if fallback.empty:
        cold_fallback_rows.append({
            "category": CATEGORY_LABEL,
            "category_id": CATEGORY_ID,
            "condition_name": condition_name,
            "candidate_pool_depth": PRIMARY_POOL_DEPTH,
            "fallback_row_count": 0,
            "fallback_case_count": 0,
            "maximum_score_error": 0.0,
            "maximum_candidate_score_error": 0.0,
            "candidate_identity_exact": True,
            "rank_identity_exact": True,
            "label_identity_exact": True,
            "fallback_identity_passed": True,
            "s2q_oof_path": str(s2q_oof_path),
        })
        continue
    current = fallback[[
        "query_id", "candidate_item_id", "candidate_rank", "candidate_score",
        "transformer_score", "rerank_rank", "label",
    ]].copy()
    merged = current.merge(
        s2q_oof_df[[
            "query_id", "candidate_item_id", "original_candidate_rank",
            "original_candidate_score", "rerank_score", "rerank_rank", "is_gt",
        ]].rename(columns={
            "rerank_rank": "s2q_rerank_rank",
            "is_gt": "s2q_label",
        }),
        on=["query_id", "candidate_item_id"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    candidate_identity_exact = bool(len(merged) == len(current) and merged["_merge"].eq("both").all())
    if not candidate_identity_exact:
        maximum_score_error = np.inf
        maximum_candidate_score_error = np.inf
        rank_identity_exact = False
        label_identity_exact = False
    else:
        maximum_score_error = float(np.max(np.abs(
            pd.to_numeric(merged["transformer_score"], errors="raise").to_numpy(dtype=float)
            - pd.to_numeric(merged["rerank_score"], errors="raise").to_numpy(dtype=float)
        )))
        maximum_candidate_score_error = float(np.max(np.abs(
            pd.to_numeric(merged["candidate_score"], errors="raise").to_numpy(dtype=float)
            - pd.to_numeric(merged["original_candidate_score"], errors="raise").to_numpy(dtype=float)
        )))
        rank_identity_exact = bool(
            pd.to_numeric(merged["candidate_rank"], errors="raise").astype(int).eq(
                pd.to_numeric(merged["original_candidate_rank"], errors="raise").astype(int)
            ).all()
            and pd.to_numeric(merged["rerank_rank"], errors="raise").astype(int).eq(
                pd.to_numeric(merged["s2q_rerank_rank"], errors="raise").astype(int)
            ).all()
        )
        label_identity_exact = bool(
            pd.to_numeric(merged["label"], errors="raise").astype(int).eq(
                pd.to_numeric(merged["s2q_label"], errors="raise").astype(int)
            ).all()
        )
    passed = bool(
        candidate_identity_exact
        and rank_identity_exact
        and label_identity_exact
        and maximum_score_error <= FALLBACK_IDENTITY_ATOL
        and maximum_candidate_score_error <= FALLBACK_IDENTITY_ATOL
    )
    cold_fallback_rows.append({
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "condition_name": condition_name,
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fallback_row_count": int(len(fallback)),
        "fallback_case_count": int(fallback["case_id"].nunique()),
        "maximum_score_error": maximum_score_error,
        "maximum_candidate_score_error": maximum_candidate_score_error,
        "candidate_identity_exact": candidate_identity_exact,
        "rank_identity_exact": rank_identity_exact,
        "label_identity_exact": label_identity_exact,
        "fallback_identity_passed": passed,
        "s2q_oof_path": str(s2q_oof_path),
    })
    if not passed:
        raise RuntimeError(
            f"{condition_name} cold fallback identity check failed: "
            f"candidate_identity_exact={candidate_identity_exact}, "
            f"rank_identity_exact={rank_identity_exact}, "
            f"label_identity_exact={label_identity_exact}, "
            f"max_score_error={maximum_score_error}, "
            f"max_candidate_score_error={maximum_candidate_score_error}, "
            f"fallback_rows={len(fallback)}"
        )

cold_fallback_qc_df = pd.DataFrame(cold_fallback_rows)

print(regime_performance_df.loc[
    regime_performance_df["prediction_scope"].eq("all_scored_rows")
    & regime_performance_df["regime"].isin(["overall", "strong"])
])


              category category_id condition_name  candidate_pool_depth  \
0   Herbal Supplements      herbal           Full                  1000   
2   Herbal Supplements      herbal           Full                  1000   
7   Herbal Supplements      herbal           P2-P                  1000   
9   Herbal Supplements      herbal           P2-P                  1000   
14  Herbal Supplements      herbal           P2-Q                  1000   
16  Herbal Supplements      herbal           P2-Q                  1000   

   prediction_scope user_scope   regime  case_count  target_in_pool_rate  \
0   all_scored_rows    overall  overall        1968             0.571646   
2   all_scored_rows     regime   strong         656             0.641768   
7   all_scored_rows    overall  overall        1968             0.560976   
9   all_scored_rows     regime   strong         656             0.631098   
14  all_scored_rows    overall  overall        1968             0.560976   
16  all_scored_row

In [10]:
# ==== RankP/Full Pool Alignment and Metadata Reconstruction ====
baseline_pool = reranked_primary["P2-P"].copy()
personalized_pool = reranked_primary["Full"].copy()

baseline_cases = set(baseline_pool["case_id"].astype(str))
personalized_cases = set(personalized_pool["case_id"].astype(str))
if baseline_cases != personalized_cases:
    raise RuntimeError("Baseline and personalized pool case universes differ.")

pool_overlap_rows = []
for case_id in sorted(baseline_cases):
    base_case = baseline_pool.loc[baseline_pool["case_id"].eq(case_id)]
    full_case = personalized_pool.loc[personalized_pool["case_id"].eq(case_id)]
    base_items = set(base_case["candidate_item_id"].astype(str))
    full_items = set(full_case["candidate_item_id"].astype(str))
    intersection = base_items & full_items
    union = base_items | full_items
    base_target = bool(base_case["label"].astype(int).max())
    full_target = bool(full_case["label"].astype(int).max())
    pool_overlap_rows.append({
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "case_id": case_id,
        "query_id": str(base_case["query_id"].iloc[0]),
        "user_id": str(base_case["user_id"].iloc[0]),
        "regime": str(base_case["regime"].iloc[0]),
        "baseline_candidate_count": int(len(base_items)),
        "personalized_candidate_count": int(len(full_items)),
        "intersection_count": int(len(intersection)),
        "union_count": int(len(union)),
        "jaccard_similarity": float(len(intersection) / len(union)) if union else np.nan,
        "baseline_only_count": int(len(base_items - full_items)),
        "personalized_only_count": int(len(full_items - base_items)),
        "target_in_baseline_pool": base_target,
        "target_in_personalized_pool": full_target,
        "target_added_by_personalized_pool": bool((not base_target) and full_target),
        "target_lost_by_personalized_pool": bool(base_target and (not full_target)),
        "analysis_role": "candidate_source_distribution_shift_not_prior_feature_effect",
        "causal_claim": False,
    })

pool_overlap_df = pd.DataFrame(pool_overlap_rows)

EXACT_ITEM_DIAGNOSTIC_COLUMNS = {
    "user_item_seen_strength",
    "user_item_recency_days",
    "user_item_recent_count_180d",
}
shift_feature_candidates = [
    "candidate_rank",
    "candidate_score",
    "candidate_score_norm_pool",
    "query_item_structured_match",
    "user_item_affinity",
]
shift_features = [
    feature for feature in shift_feature_candidates
    if feature in baseline_pool.columns and feature in personalized_pool.columns
]
if set(shift_features).intersection(EXACT_ITEM_DIAGNOSTIC_COLUMNS):
    raise RuntimeError("Previously-reviewed-item diagnostics entered Transformer pool-shift analysis.")
if not shift_features:
    raise RuntimeError("No verified common feature is available for pool distribution-shift diagnostics.")

distribution_shift_rows = []
for feature in shift_features:
    base_case_values = (
        baseline_pool.assign(_value=pd.to_numeric(baseline_pool[feature], errors="coerce"))
        .groupby("case_id", as_index=False, observed=True)["_value"].mean()
        .rename(columns={"_value": "baseline_case_mean"})
    )
    full_case_values = (
        personalized_pool.assign(_value=pd.to_numeric(personalized_pool[feature], errors="coerce"))
        .groupby("case_id", as_index=False, observed=True)["_value"].mean()
        .rename(columns={"_value": "personalized_case_mean"})
    )
    paired = base_case_values.merge(full_case_values, on="case_id", how="inner", validate="one_to_one").dropna()
    if len(paired) != len(baseline_cases):
        raise RuntimeError(f"Pool-shift feature {feature} lacks complete case-level coverage.")
    differences = paired["personalized_case_mean"] - paired["baseline_case_mean"]
    pooled_sd = math.sqrt(
        (float(paired["baseline_case_mean"].var(ddof=0)) + float(paired["personalized_case_mean"].var(ddof=0))) / 2.0
    )
    distribution_shift_rows.append({
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "feature": feature,
        "analysis_grain": "case_level_candidate_mean",
        "case_count": int(len(paired)),
        "baseline_mean_of_case_means": float(paired["baseline_case_mean"].mean()),
        "personalized_mean_of_case_means": float(paired["personalized_case_mean"].mean()),
        "paired_mean_difference_personalized_minus_baseline": float(differences.mean()),
        "paired_median_difference_personalized_minus_baseline": float(differences.median()),
        "standardized_mean_difference": float(differences.mean() / pooled_sd) if pooled_sd > 0 else np.nan,
        "candidate_source_effect_label": "Full_minus_P2-P_personalized_candidate_source",
        "not_a_user_prior_feature_effect": True,
        "causal_claim": False,
    })

pool_distribution_shift_df = pd.DataFrame(distribution_shift_rows)

print(pool_overlap_df[[
    "jaccard_similarity", "target_added_by_personalized_pool", "target_lost_by_personalized_pool"
]].mean(numeric_only=True))


jaccard_similarity                   0.928651
target_added_by_personalized_pool    0.016768
target_lost_by_personalized_pool     0.006098
dtype: float64


In [15]:
# ==== Epoch-History Extraction and Learning Curves ====
def epoch_history_from_payload(condition_name, fold_id, payload):
    history = None
    history_key = None
    for key in ("training_history", "epoch_history", "history"):
        candidate = payload.get(key)
        if isinstance(candidate, list) and candidate and all(isinstance(row, dict) for row in candidate):
            history = candidate
            history_key = key
            break
    rows = []
    if history is not None:
        for row in history:
            normalized = {str(key): value for key, value in row.items()}
            epoch_value = normalized.get("epoch", normalized.get("epoch_idx"))
            if epoch_value is None:
                continue
            train_loss = normalized.get("train_loss", normalized.get("training_loss"))
            valid_loss = normalized.get("valid_loss", normalized.get("validation_loss"))
            ndcg5 = None
            for key in ("NDCG@5", "ndcg_at_5", "ndcg5", "valid_ndcg_at_5"):
                if key in normalized:
                    ndcg5 = normalized[key]
                    break
            rows.append({
                "category": CATEGORY_LABEL,
                "category_id": CATEGORY_ID,
                "condition_name": condition_name,
                "candidate_pool_depth": PRIMARY_POOL_DEPTH,
                "fold_id": int(fold_id),
                "epoch": int(epoch_value),
                "train_loss": pd.to_numeric(train_loss, errors="coerce"),
                "validation_loss": pd.to_numeric(valid_loss, errors="coerce"),
                "validation_ndcg_at_5": pd.to_numeric(ndcg5, errors="coerce"),
                "source_key": history_key,
            })
    return rows


if "condition_fold_contracts" not in globals():
    raise RuntimeError(
        "Run the held-out/model-contract validation cell before checkpoint payload extraction; "
        "it defines condition_fold_contracts."
    )

checkpoint_payload_rows = []
epoch_rows = []
checkpoint_contract_sources = {
    "P2-Q": "P2-Q",
    "P2-P": "P2-P" if "P2-P" in condition_fold_contracts else "Full",
}
for condition_name, contract_condition in checkpoint_contract_sources.items():
    for fold_id, fold_info in sorted(condition_fold_contracts[contract_condition].items()):
        if bool(fold_info.get("fallback_used", False)):
            checkpoint_payload_rows.append({
                "condition_name": condition_name,
                "fold_id": int(fold_id),
                "checkpoint_path": str(fold_info.get("model_path", "")),
                "checkpoint_fallback_used": True,
                "epoch_history_exported": False,
                "checkpoint_keys_json": "[]",
            })
            continue
        checkpoint_path = as_path(fold_info.get("model_path"), "checkpoint diagnostics")
        payload = safe_torch_load(checkpoint_path)
        extracted = epoch_history_from_payload(condition_name, fold_id, payload)
        epoch_rows.extend(extracted)
        checkpoint_payload_rows.append({
            "condition_name": condition_name,
            "fold_id": int(fold_id),
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_fallback_used": False,
            "epoch_history_exported": bool(extracted),
            "checkpoint_keys_json": json.dumps(sorted(payload.keys()), ensure_ascii=False),
        })

epoch_diagnostics_df = pd.DataFrame(epoch_rows, columns=[
    "category", "category_id", "condition_name", "candidate_pool_depth", "fold_id",
    "epoch", "train_loss", "validation_loss", "validation_ndcg_at_5", "source_key",
])
checkpoint_payload_df = pd.DataFrame(checkpoint_payload_rows)

cv_frames = []
for condition_name in ("P2-Q", "P2-P"):
    manifest = run_manifests[condition_name]
    cv_path_value = manifest.get("output_paths", {}).get("model_cv_summary")
    if not cv_path_value:
        continue
    cv_path = Path(str(cv_path_value))
    if not cv_path.exists():
        raise FileNotFoundError(f"Missing {condition_name} model CV summary: {cv_path}")
    cv_frame = pd.read_csv(cv_path)
    require_columns(cv_frame, ["pool_depth", "fold"], f"{condition_name} model CV summary")
    cv_frame = cv_frame.loc[
        pd.to_numeric(cv_frame["pool_depth"], errors="raise").astype(int).eq(PRIMARY_POOL_DEPTH)
    ].copy()
    cv_frame["condition_name"] = condition_name
    cv_frame["fold_id"] = pd.to_numeric(cv_frame["fold"], errors="raise").astype(int)
    cv_frame["cv_summary_path"] = str(cv_path)
    cv_frames.append(cv_frame)

cv_summary_df = pd.concat(cv_frames, ignore_index=True) if cv_frames else pd.DataFrame()
checkpoint_diagnostics_df = checkpoint_payload_df.copy()
if not cv_summary_df.empty:
    keep_columns = [
        column for column in [
            "condition_name", "fold_id", "best_epoch", "best_valid_loss",
            "fallback_used", "ranking_loss", "n_fit_queries_positive",
            "n_valid_queries_positive", "n_test_queries", "cv_summary_path",
        ]
        if column in cv_summary_df.columns
    ]
    value_columns = [
        column for column in keep_columns if column not in {"condition_name", "fold_id", "cv_summary_path"}
    ]
    if value_columns:
        inconsistent = (
            cv_summary_df.groupby(["condition_name", "fold_id"], observed=True)[value_columns]
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
        )
        if inconsistent.any():
            raise RuntimeError("Model CV summary contains conflicting duplicate fold diagnostics.")
    cv_compact = cv_summary_df[keep_columns].drop_duplicates(["condition_name", "fold_id"])
    checkpoint_diagnostics_df = checkpoint_diagnostics_df.merge(
        cv_compact,
        on=["condition_name", "fold_id"],
        how="left",
        validate="one_to_one",
    )

selected_overall = selected_checkpoint_performance_df.loc[
    selected_checkpoint_performance_df["user_scope"].eq("overall")
].copy()
checkpoint_diagnostics_df = checkpoint_diagnostics_df.merge(
    selected_overall[[
        "condition_name", "fold_id", "case_count", "selected_checkpoint_heldout_ndcg_at_5"
    ]],
    on=["condition_name", "fold_id"],
    how="left",
    validate="one_to_one",
)
checkpoint_diagnostics_df.insert(0, "category", CATEGORY_LABEL)
checkpoint_diagnostics_df.insert(1, "category_id", CATEGORY_ID)
checkpoint_diagnostics_df.insert(3, "candidate_pool_depth", PRIMARY_POOL_DEPTH)
checkpoint_diagnostics_df["checkpoint_selection_metric"] = "validation_ndcg_at_5"
checkpoint_diagnostics_df["checkpointing_methodological_assessment"] = (
    "checkpoint_selected_by_max_validation_ndcg_at_5_then_min_validation_ce_then_earliest_epoch"
)
checkpoint_diagnostics_df["selected_checkpoint_heldout_ndcg_is_epoch_selection_metric"] = False

epoch_available = not epoch_diagnostics_df.empty
epoch_diagnostics_availability_df = pd.DataFrame([
    {
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "diagnostic": "per_epoch_training_loss",
        "available": bool(epoch_available and epoch_diagnostics_df["train_loss"].notna().any()),
        "available_summary": "best checkpoint only" if not epoch_available else "checkpoint history",
        "reason_if_unavailable": "13a/13b checkpoints and manifests do not export per-epoch history; no retraining is performed." if not epoch_available else "",
    },
    {
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "diagnostic": "per_epoch_validation_loss",
        "available": bool(epoch_available and epoch_diagnostics_df["validation_loss"].notna().any()),
        "available_summary": "best_valid_loss and best_epoch are retained in the fold CV summary",
        "reason_if_unavailable": "Only the selected best validation loss is exported; no per-epoch series is regenerated." if not epoch_available else "",
    },
    {
        "category": CATEGORY_LABEL,
        "category_id": CATEGORY_ID,
        "diagnostic": "per_epoch_validation_ndcg_at_5",
        "available": bool(epoch_available and epoch_diagnostics_df["validation_ndcg_at_5"].notna().any()),
        "available_summary": "validation NDCG@5 is the checkpoint-selection metric; selected-checkpoint held-out performance is diagnostic only",
        "reason_if_unavailable": "Per-epoch NDCG@5 history is unavailable, but the exported selected-checkpoint metadata must still identify validation_ndcg_at_5." if not epoch_available else "",
    },
])

MASKING_INTERPRETATION_LABEL = (
    "descriptive_predictive_dependence_from_heldout_group_mean_masking_not_causal_contribution"
)
for _masking_frame in [
    masking_by_fold_df,
    masking_summary_df,
    primary_semantic_group_summary_df,
    primary_facet_family_summary_df,
    branch_contribution_df,
    cross_model_summary_df,
]:
    _masking_frame["interpretation_label"] = MASKING_INTERPRETATION_LABEL
    _masking_frame["causal_contribution_claim"] = False

selected_checkpoint_performance_df["performance_authority"] = (
    "diagnostic_checkpoint_reproduction_not_canonical_pipeline_performance"
)
margin_diagnostics_df["performance_authority"] = (
    "diagnostic_rank_reconstruction_not_canonical_pipeline_performance"
)

qc_rows = []
if "heldout_reranked_prediction_qc_df" in globals() and not heldout_reranked_prediction_qc_df.empty:
    for row in heldout_reranked_prediction_qc_df.itertuples(index=False):
        qc_rows.append({
            "check": "heldout_oof_vs_reranked_score_consistency",
            "severity": "warning",
            "passed": bool(row.passed),
            "condition_name": row.condition_name,
            "candidate_pool_depth": int(row.candidate_pool_depth),
            "fold_id": np.nan,
            "details": (
                "Nonblocking audit: held-out oof_prediction is authoritative for Notebook 19; "
                f"score_mismatch_rows={row.mismatch_count}; "
                f"rank_mismatch_rows={getattr(row, 'rerank_rank_mismatch_count', 0)}; "
                f"max_abs_error={row.maximum_absolute_error}; "
                f"policy={row.score_policy}"
            ),
        })
for row in source_availability_df.itertuples(index=False):
    _is_explicit_s2p_scope_reduction = (
        row.condition_name == "P2-P"
        and row.interpretation_path_mode == "path_B_explicit_scope_reduction"
    )
    qc_rows.append({
        "check": f"{row.condition_name}_source_artifacts",
        "severity": "warning" if _is_explicit_s2p_scope_reduction else "error",
        "passed": bool(row.interpretation_manifest_available),
        "condition_name": row.condition_name,
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": (
            f"{row.analysis_policy}; mode={row.interpretation_path_mode}; "
            f"reason={row.scope_reduction_reason}"
        ),
    })
for row in reproduction_qc_df.itertuples(index=False):
    scoring_available = bool(getattr(row, "interpretation_scoring_available", row.prediction_reproduced))
    ndcg_reproduced = bool(getattr(row, "ndcg_at_5_reproduced", row.prediction_reproduced))
    top5_mismatch = int(getattr(row, "top5_status_mismatch_count", 0))

    qc_rows.append({
        "check": "native_fold_interpretation_scoring_available",
        "severity": "error",
        "passed": scoring_available,
        "condition_name": row.condition_name,
        "candidate_pool_depth": int(row.candidate_pool_depth),
        "fold_id": int(row.fold_id),
        "details": (
            f"native_rows={row.native_transformer_row_count}; "
            f"cold_fallback_excluded={row.cold_fallback_row_count_excluded}; "
            f"model_fallback_excluded={row.model_level_fallback_row_count_excluded}; "
            f"max_abs_error={row.maximum_absolute_native_export_error}"
        ),
    })

    qc_rows.append({
        "check": "native_fold_prediction_reproduction",
        "severity": "warning",
        "passed": bool(row.prediction_reproduced and ndcg_reproduced and top5_mismatch == 0),
        "condition_name": row.condition_name,
        "candidate_pool_depth": int(row.candidate_pool_depth),
        "fold_id": int(row.fold_id),
        "details": (
            "Nonblocking audit: exported oof_prediction is authoritative for held-out baseline; "
            f"raw_reproduced={row.prediction_reproduced}; "
            f"ndcg_reproduced={ndcg_reproduced}; "
            f"target_rank_mismatch={getattr(row, 'target_rank_mismatch_count', 0)}; "
            f"top5_status_mismatch={top5_mismatch}; "
            f"max_abs_error={row.maximum_absolute_native_export_error}; "
            f"max_ndcg_error={getattr(row, 'maximum_absolute_ndcg_at_5_error', np.nan)}"
        ),
    })
for row in cold_fallback_qc_df.itertuples(index=False):
    qc_rows.append({
        "check": "strict_cold_exact_s2q_oof_fallback_identity",
        "severity": "error",
        "passed": bool(row.fallback_identity_passed),
        "condition_name": row.condition_name,
        "candidate_pool_depth": int(row.candidate_pool_depth),
        "fold_id": np.nan,
        "details": f"fallback_rows={row.fallback_row_count}; max_score_error={row.maximum_score_error}",
    })
for row in fold_lineage_rows:
    qc_rows.append({
        "check": "user_group_disjoint_oof_and_fold_lineage",
        "severity": "error",
        "passed": bool(row["user_group_disjoint"] and row["heldout_fold_assignment_exact"]),
        "condition_name": row["condition_name"],
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": (
            f"queries={row['query_count_checked']}; users={row['user_count_checked']}; "
            f"assignment={row['assignment_path']}"
        ),
    })
qc_rows.extend([
    {
        "check": "primary_depth_1000",
        "severity": "error",
        "passed": bool(PRIMARY_POOL_DEPTH == HEADLINE_REPORT_POOL_DEPTH),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "All primary interpretation, margin, regime, and shift outputs use depth 1000.",
    },
    {
        "check": "overall_and_strong_reported",
        "severity": "error",
        "passed": bool(
            set(regime_performance_df.loc[
                regime_performance_df["prediction_scope"].eq("all_scored_rows"), "regime"
            ]) >= {"overall", "strong"}
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "Overall and Strong are present without pooling categories.",
    },
    {
        "check": "full_shared_model_lineage",
        "severity": "error",
        "passed": bool(
            (
                run_manifests["Full"].get("shared_all_prior_model_role") == "load_and_score"
                and canonical_condition_token(
                    interpretation_manifests["Full"].get("model_source_condition")
                ) == "P2-P"
            )
            or (
                run_manifests["Full"].get("shared_all_prior_model_role") == "train_and_export"
                and canonical_condition_token(
                    interpretation_manifests["Full"].get("model_source_condition")
                ) == "Full"
            )
        ),
        "condition_name": "Full",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "13c Full is accepted as either sealed load-and-score or legacy trained-Full upstream.",
    },
    {
        "check": "checkpoint_selection_metric",
        "severity": "error",
        "passed": bool(
            all(
                run_manifests[condition].get("checkpoint_selection_metric")
                == "validation_ndcg_at_5"
                for condition in SOURCE_DIRS
            )
            and checkpoint_diagnostics_df["checkpoint_selection_metric"]
            .eq("validation_ndcg_at_5").all()
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "Checkpoint selection is max validation NDCG@5, then min validation CE, then earliest epoch.",
    },
    {
        "check": "full_interpretation_category_mismatch_disabled_false",
        "severity": "error",
        "passed": bool(
            (
                run_manifests["Full"].get("shared_all_prior_model_role") == "load_and_score"
                and interpretation_manifests["Full"].get("disabled_category_mismatch") is False
            )
            or run_manifests["Full"].get("shared_all_prior_model_role") == "train_and_export"
        ),
        "condition_name": "Full",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "disabled_category_mismatch=false is required for load-and-score; legacy trained-Full is accepted with its historical flag value.",
    },
    {
        "check": "canonical_performance_from_notebook14",
        "severity": "error",
        "passed": bool(
            not notebook14_canonical_metrics_df.empty
            and canonical_source_qc_df["canonical_source_passed"].all()
            and regime_performance_df["performance_authority"]
            .eq("Notebook14_pipeline_canonical_per_case_metrics").all()
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": f"Canonical performance source: {notebook14_canonical_metrics_path}",
    },
    {
        "check": "masking_outputs_regenerated_and_noncausal",
        "severity": "error",
        "passed": bool(
            not masking_by_fold_df.empty
            and not masking_summary_df.empty
            and masking_by_fold_df["interpretation_label"].eq(MASKING_INTERPRETATION_LABEL).all()
            and not masking_by_fold_df["causal_contribution_claim"].any()
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "Masking values are descriptive predictive dependence, not causal contributions.",
    },
    {
        "check": "noncausal_interpretation_boundary",
        "severity": "error",
        "passed": bool(
            not masking_by_fold_df["causal_claim"].any()
            and not masking_by_fold_df["causal_contribution_claim"].any()
            and not pool_distribution_shift_df["causal_claim"].any()
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "Masking, margins, and distribution shift are predictive/descriptive evidence only.",
    },
])
qc_rows.extend([
    {
        "check": "s2p_interpretation_coverage_declared",
        "severity": "error",
        "passed": bool(
            ("P2-P" in interpretation_manifests)
            or (
                s2p_interpretation_mode == "path_B_explicit_scope_reduction"
                and bool(s2p_interpretation_reason)
                and not interpretation_scope_note_df.empty
            )
        ),
        "condition_name": "P2-P",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": f"mode={s2p_interpretation_mode}; reason={s2p_interpretation_reason}",
    },
    {
        "check": "s2p_masking_coverage_matches_declared_scope",
        "severity": "error",
        "passed": bool(
            (
                "P2-P" in interpretation_manifests
                and "P2-P" in set(masking_by_fold_df["condition_name"].astype(str))
            )
            or (
                "P2-P" not in interpretation_manifests
                and "P2-P" not in set(masking_by_fold_df["condition_name"].astype(str))
                and s2p_interpretation_mode == "path_B_explicit_scope_reduction"
            )
        ),
        "condition_name": "P2-P",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "RankP masking is either fully validated or explicitly excluded with a thesis-facing scope note.",
    },
    {
        "check": "refreshed_notebook14_15_16_chain",
        "severity": "error",
        "passed": bool(
            analysis_chain_preflight.get("notebook14_ready")
            and analysis_chain_preflight.get("notebook15_ready")
            and analysis_chain_preflight.get("notebook16_ready")
        ),
        "condition_name": "all",
        "candidate_pool_depth": PRIMARY_POOL_DEPTH,
        "fold_id": np.nan,
        "details": "Execution order enforced: 13c -> 14 -> 15 -> 16 -> 19.",
    },
])
qc_df = pd.DataFrame(qc_rows)
failed_error_checks = qc_df.loc[qc_df["severity"].eq("error") & ~qc_df["passed"]]
if not failed_error_checks.empty:
    raise RuntimeError(f"Transformer interpretation QC failed:\n{failed_error_checks}")

output_frames = {
    "source_availability": source_availability_df,
    "interpretation_scope_note": interpretation_scope_note_df,
    "canonical_source_qc": canonical_source_qc_df,
    "semantic_group_definition": semantic_group_definition_df,
    "branch_group_definition": branch_group_definition_df,
    "facet_family_mapping": facet_family_mapping_df,
    "reproduction_qc": reproduction_qc_df,
    "masking_by_fold": masking_by_fold_df,
    "masking_summary": masking_summary_df,
    "semantic_group_stability": semantic_group_stability_df,
    "semantic_group_fold_rank_correlation": semantic_group_fold_rank_correlation_df,
    "primary_semantic_group_summary": primary_semantic_group_summary_df,
    "primary_facet_family_summary": primary_facet_family_summary_df,
    "branch_contribution": branch_contribution_df,
    "cross_model_summary": cross_model_summary_df,
    "margin_diagnostics": margin_diagnostics_df,
    "top5_taxonomy": top5_taxonomy_df,
    "regime_performance": regime_performance_df,
    "pool_overlap": pool_overlap_df,
    "pool_distribution_shift": pool_distribution_shift_df,
    "checkpoint_diagnostics": checkpoint_diagnostics_df,
    "selected_checkpoint_performance": selected_checkpoint_performance_df,
    "epoch_diagnostics": epoch_diagnostics_df,
    "epoch_diagnostics_availability": epoch_diagnostics_availability_df,
    "cold_fallback_qc": cold_fallback_qc_df,
    "qc": qc_df,
}
NON_FRAME_OUTPUT_KEYS = {"manifest", "reconstructed_s2p_manifest"}

if set(output_frames) != set(OUTPUT_FILES) - NON_FRAME_OUTPUT_KEYS:
    raise RuntimeError(
        "Output frame keys and OUTPUT_FILES are inconsistent: "
        f"missing={sorted((set(OUTPUT_FILES) - NON_FRAME_OUTPUT_KEYS) -
        set(output_frames))}, "
        f"extra={sorted(set(output_frames) - (set(OUTPUT_FILES) -
        NON_FRAME_OUTPUT_KEYS))}"
    )
# SC-2 pre-write containment check for every Notebook 19 output.
for _output_path in OUTPUT_FILES.values():
    try:
        Path(_output_path).resolve().relative_to(OUT_DIR.resolve())
    except ValueError as exc:
        raise RuntimeError(f"Notebook 19 output path is outside OUT_DIR: {_output_path}") from exc
for name, frame in output_frames.items():
    require_unique_columns(frame, f"output {name}")
    frame.to_csv(OUTPUT_FILES[name], index=False, encoding="utf-8-sig")

_sc2_source_sha256_by_condition = {}
for _row in source_availability_df.to_dict(orient="records"):
    _condition = str(_row["condition_name"])
    _sc2_source_sha256_by_condition[_condition] = {}
    for _field in ["run_manifest_path", "reranked_candidates_path", "interpretation_manifest_path"]:
        _path = Path(str(_row.get(_field, "")))
        _sc2_source_sha256_by_condition[_condition][_field] = str(_path)
        _sc2_source_sha256_by_condition[_condition][_field.replace("_path", "_sha256")] = (
            file_sha256(_path) if _path.exists() else ""
        )

_sc2_reconstructed_manifest_path = Path(OUTPUT_FILES["reconstructed_s2p_manifest"])
_sc2_reconstructed_manifest_sha256 = (
    file_sha256(_sc2_reconstructed_manifest_path)
    if _sc2_reconstructed_manifest_path.exists()
    else ""
)
try:
    _sc2_reconstructed_manifest_path.resolve().relative_to(OUT_DIR.resolve())
except ValueError as exc:
    raise RuntimeError("Reconstructed RankP interpretation manifest is outside Notebook 19 OUT_DIR.") from exc
for _output_path in OUTPUT_FILES.values():
    try:
        Path(_output_path).resolve().relative_to(OUT_DIR.resolve())
    except ValueError as exc:
        raise RuntimeError(f"Notebook 19 output path is outside OUT_DIR: {_output_path}") from exc

analysis_manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "latest_revision": "Batch 2 refresh 2026-07-21; depth-scope disclosure corrected to executed five-depth NB13b record and SC-2 identifier repair 2026-07-26",
    "category": CATEGORY_LABEL,
    "category_id": CATEGORY_ID,
    "model_family": MODEL_FAMILY,
    "primary_metric": "NDCG@5",
    "primary_report_pool_depth": PRIMARY_POOL_DEPTH,
    "headline_report_pool_depth": HEADLINE_REPORT_POOL_DEPTH,
    "interpretation_depth_scope": INTERPRETATION_DEPTH_SCOPE,
    "depth_scope_disclosure": (
        "Interpretation diagnostics are computed at pool depth 1000, the same "
        "depth as the reporting headline (audit rev2 D19-4). Notebook 13b "
        "materialises interpretation artefacts at all five depths "
        "{100, 300, 500, 700, 1000}; depth 1000 is an alignment choice with "
        "the headline, not an availability constraint. K=700 appears only as "
        "the appendix bounded-reranking scenario."
    ),
    "source_notebooks": SOURCE_NOTEBOOKS,
    "source_run_manifests": {
        condition: str(SOURCE_DIRS[condition] / "run_manifest.json") for condition in SOURCE_DIRS
    },
    "source_interpretation_manifests": {
        condition: str(path) for condition, path in interpretation_manifest_paths.items()
    },
    "source_availability": source_availability_df.to_dict(orient="records"),
    "s2p_interpretation_path": s2p_interpretation_mode,
    "upstream_artifacts_modified": False,
    "original_source_paths_and_sha256": _sc2_source_sha256_by_condition,
    "reconstructed_s2p_manifest_path": str(_sc2_reconstructed_manifest_path),
    "reconstructed_s2p_manifest_sha256": _sc2_reconstructed_manifest_sha256,
    "s2p_interpretation_scope_reason": s2p_interpretation_reason,
    "s2p_masking_available": bool("P2-P" in interpretation_manifests),
    "thesis_facing_scope_note": interpretation_scope_note_df.to_dict(orient="records"),
    "analysis_chain_preflight": analysis_chain_preflight,
    "heldout_policy": "each fold checkpoint scores only its assigned native OOF rows at depth 1000",
    "fold_lineage_qc": fold_lineage_rows,
    "native_all_prior_prediction_source": NATIVE_ALL_PRIOR_SOURCE,
    "native_full_prediction_source": FULL_NATIVE_SOURCE,
    "native_no_prior_prediction_source": NO_PRIOR_NATIVE_SOURCE,
    "native_prediction_sources_by_condition": {
        condition: sorted(native_prediction_sources_for_condition(condition))
        for condition in interpretation_manifests
    },
    "excluded_all_prior_prediction_source": FALLBACK_SOURCE,
    "cold_fallback_policy": "validate exact same-category 13a Base OOF identity; exclude from native All Prior reproduction and masking",
    "reproduction_atol": REPRODUCTION_ATOL,
    "fallback_identity_atol": FALLBACK_IDENTITY_ATOL,
    "masking_policy": "set selected standardized structured inputs to training-fold mean (zero); mask rank embedding for retrieval_rank and candidate_common",
    "semantic_groups": REQUIRED_SEMANTIC_GROUPS,
    "branch_groups": REQUIRED_BRANCH_GROUPS,
    "brand_distinct_from_functional_facets": True,
    "group_definitions_selected_without_outcomes": True,
    "labels_used_only_for_posthoc_metrics": True,
    "target_or_future_interactions_used_for_prior_construction": False,
    "model_training_or_checkpoint_writes_performed": False,
    "upstream_13c_modified_and_rerun_for_batch2": True,
    "upstream_13c_retrained": bool(run_manifests["Full"].get("shared_all_prior_model_role") == "train_and_export"),
    "full_model_lineage": ("13c Full loads and scores 13b shared All Prior fold checkpoints" if run_manifests["Full"].get("shared_all_prior_model_role") == "load_and_score" else "13c Full legacy trained-Full upstream"),
    "checkpoint_selection_metric": "validation_ndcg_at_5",
    "checkpointing_assessment": "max_validation_ndcg_at_5_then_min_validation_ce_then_earliest_epoch",
    "canonical_performance_source": {
        "notebook": "14",
        "per_case_metrics_path": str(notebook14_canonical_metrics_path),
        "manifest_path": str(notebook14_canonical_manifest_path),
        "authority": "sole_canonical_pipeline_performance_source",
    },
    "disabled_category_mismatch": False,
    "stale_analysis_outputs_removed_before_refresh": True,
    "masking_outputs_regenerated": True,
    "masking_interpretation_label": MASKING_INTERPRETATION_LABEL,
    "masking_values_are_causal_contributions": False,
    "epoch_diagnostics_availability": epoch_diagnostics_availability_df.to_dict(orient="records"),
    "margin_definition": {
        "target_score_margin": "target score minus mean negative-candidate score",
        "target_vs_top_negative_margin": "target score minus maximum negative-candidate score",
    },
    "top5_taxonomy": [
        "target_absent_from_pool", "top5_entry", "top5_loss", "top5_retained", "remains_below_top5"
    ],
    "pool_shift_contrast": "Full minus P2-P is a personalized candidate-source diagnostic, not a user-prior feature effect",
    "overall_and_strong_reported": True,
    "raw_cases_pooled_across_categories": False,
    "interpretation_evidence_role": "descriptive_predictive_dependence_noncausal_not_canonical_performance",
    "attention_weights_used_as_explanations": False,
    "output_files": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_FILES["manifest"].write_text(
    json.dumps(analysis_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Saved Transformer held-out interpretation outputs:")
for name, path in OUTPUT_FILES.items():
    print(f"- {name}: {path}")


Saved Transformer held-out interpretation outputs:
- source_availability: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/transformer_source_availability.csv
- reconstructed_s2p_manifest: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/reconstructed_s2p_feature_interpretation_manifest.json
- interpretation_scope_note: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/transformer_interpretation_scope_note.csv
- canonical_source_qc: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/transformer_canonical_source_qc.csv
- semantic_group_definition: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/transformer_semantic_group_definition.csv
- branch_group_definition: /content/drive/MyDrive

In [16]:
# ==== Batch C2 — Interpretation Guard (Exact-item Familiarity Exclusion) ====
# Batch C2 interpretation guard
for _condition_name, _feature_names in condition_feature_names.items():
    _exact_item_columns = {
        "user_item_seen_strength",
        "user_item_recency_days",
        "user_item_recent_count_180d",
    }
    _leak = sorted(set(_feature_names).intersection(_exact_item_columns))
    if _leak:
        raise RuntimeError(
            f"{_condition_name} Transformer interpretation contains previously-reviewed-item diagnostics: {_leak}"
        )
if set(shift_features).intersection(_exact_item_columns):
    raise RuntimeError("Transformer distribution-shift diagnostics include previously-reviewed-item columns.")
print("Transformer primary interpretation seen-item exclusion QC passed.")

if not checkpoint_diagnostics_df["checkpoint_selection_metric"].eq(
    "validation_ndcg_at_5"
).all():
    raise RuntimeError("Stale validation_loss checkpoint-selection metadata remains.")
_full_role = str(run_manifests["Full"].get("shared_all_prior_model_role", "train_and_export"))
if _full_role == "load_and_score" and interpretation_manifests["Full"].get("disabled_category_mismatch") is not False:
    raise RuntimeError("Full load-and-score interpretation manifest still carries disabled-category mismatch flag.")
if not regime_performance_df["performance_authority"].eq(
    "Notebook14_pipeline_canonical_per_case_metrics"
).all():
    raise RuntimeError("Notebook 19 reconstructed canonical performance outside Notebook 14.")
if masking_by_fold_df.empty or not masking_by_fold_df["interpretation_label"].eq(
    MASKING_INTERPRETATION_LABEL
).all():
    raise RuntimeError("Transformer masking outputs were not regenerated under the Batch 2 label.")
print("Batch 2 Transformer interpretation refresh QC passed.")

if not (
    ("P2-P" in interpretation_manifests and s2p_interpretation_mode.startswith("path_A"))
    or (
        "P2-P" not in interpretation_manifests
        and s2p_interpretation_mode == "path_B_explicit_scope_reduction"
        and bool(s2p_interpretation_reason)
    )
):
    raise RuntimeError("RankP interpretation coverage is neither valid Path A nor explicit Path B.")
if not all(analysis_chain_preflight.get(key) is True for key in [
    "notebook14_ready", "notebook15_ready", "notebook16_ready"
]):
    raise RuntimeError("Notebook 19 ran without a refreshed Notebook 14-16 chain.")
print("RankP interpretation path:", s2p_interpretation_mode)


Transformer primary interpretation seen-item exclusion QC passed.
Batch 2 Transformer interpretation refresh QC passed.
RankP interpretation path: path_B_explicit_scope_reduction


## SELF-CHECK SC-2 Path A/B Exception Safety

In [17]:
# ==== SELF-CHECK SC-2 — Path A/B Exception-Safety Tests ====
# SELF-CHECK SC-2: Path A / Path B exception-safety synthetic tests.
# This cell is intentionally local-only and does not touch thesis artifacts.
def _sc2_path_ab_exception_safety_synthetic_tests():
    import json
    import tempfile
    from pathlib import Path

    def _byte_state(directory: Path) -> dict[str, str]:
        state = {}
        for path in sorted(directory.rglob("*")):
            if path.is_file():
                state[str(path.relative_to(directory))] = path.read_bytes().hex()
        return state

    def _path_ab_boundary(authoritative_loader):
        try:
            payload = authoritative_loader()
            return "path_A", payload
        except S2PInterpretationUnavailable as exc:
            return "path_B", {"reason": f"{type(exc).__name__}: {exc}"}

    with tempfile.TemporaryDirectory() as tmp:
        tmp_root = Path(tmp)
        upstream_dir = tmp_root / "upstream_13b"
        upstream_dir.mkdir()
        (upstream_dir / "contract.json").write_text(
            json.dumps({"checkpoint_selection_metric": "validation_ndcg_at_5"}),
            encoding="utf-8",
        )
        before = _byte_state(upstream_dir)

        mode, payload = _path_ab_boundary(lambda: {"manifest": "valid_path_A"})
        if mode != "path_A" or payload.get("manifest") != "valid_path_A":
            raise AssertionError("Valid authoritative RankP interpretation artifacts must stay on Path A.")

        mode, payload = _path_ab_boundary(
            lambda: (_ for _ in ()).throw(S2PInterpretationUnavailable("missing authoritative artifact"))
        )
        if mode != "path_B" or "S2PInterpretationUnavailable" not in payload.get("reason", ""):
            raise AssertionError("Only expected missing/inconsistent authoritative artifacts may activate Path B.")

        try:
            _path_ab_boundary(lambda: (_ for _ in ()).throw(KeyError("programming bug")))
        except KeyError:
            pass
        else:
            raise AssertionError("Unexpected KeyError must be re-raised, not converted to Path B.")

        after = _byte_state(upstream_dir)
        if before != after:
            raise AssertionError("Notebook 19 Path A/B handling must not mutate upstream artifact directories.")

    return {
        "valid_path_a_succeeds": True,
        "missing_artifact_activates_path_b": True,
        "unexpected_keyerror_reraises": True,
        "upstream_directory_byte_unchanged": True,
    }

SC2_PATH_AB_SYNTHETIC_TEST_RESULTS = _sc2_path_ab_exception_safety_synthetic_tests()
print("SC-2 Path A/B synthetic tests passed:", SC2_PATH_AB_SYNTHETIC_TEST_RESULTS)

SC-2 Path A/B synthetic tests passed: {'valid_path_a_succeeds': True, 'missing_artifact_activates_path_b': True, 'unexpected_keyerror_reraises': True, 'upstream_directory_byte_unchanged': True}


In [18]:
# ==== Final Verification Report Export ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math

try:
    import numpy as _report_np
except Exception:
    _report_np = None
try:
    import pandas as _report_pd
except Exception:
    _report_pd = globals().get("pd")


def _report_is_dataframe(value):
    return _report_pd is not None and isinstance(value, _report_pd.DataFrame)


def _report_is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if _report_pd is not None:
        try:
            missing = _report_pd.isna(value)
            if isinstance(missing, (bool, type(None))):
                return bool(missing)
        except Exception:
            pass
    return False


def _report_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if _report_np is not None and isinstance(value, _report_np.generic):
        return _report_jsonable(value.item())
    if _report_is_missing(value):
        return None
    if isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return None if math.isnan(value) else value
    if isinstance(value, dict):
        return {str(k): _report_jsonable(v) for k, v in value.items()}
    if _report_pd is not None:
        if isinstance(value, _report_pd.Series):
            return [_report_jsonable(v) for v in value.tolist()]
        if _report_is_dataframe(value):
            return _report_frame_records(value)
    if isinstance(value, (list, tuple, set)):
        return [_report_jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        try:
            return _report_jsonable(value.tolist())
        except Exception:
            pass
    if hasattr(value, "item"):
        try:
            return _report_jsonable(value.item())
        except Exception:
            pass
    return str(value)


def _report_frame_records(frame, limit=50, columns=None, drop_case_columns=True):
    if not _report_is_dataframe(frame):
        return []
    work = frame.copy()
    if columns is not None:
        keep = [column for column in columns if column in work.columns]
        work = work[keep]
    if drop_case_columns:
        disallowed = {
            "case_id", "query_id", "user_id", "item_id", "parent_asin",
            "target_parent_asin", "target_item_id", "review_id",
        }
        drop = [column for column in work.columns if str(column).lower() in disallowed]
        if drop:
            work = work.drop(columns=drop)
    records = [_report_jsonable(row) for row in work.head(limit).to_dict(orient="records")]
    if len(work) > limit:
        records.append({"note": "truncated", "row_count": int(len(work)), "rows_emitted": int(limit)})
    return records


def _report_output_dir():
    if "OUT_DIR" in globals():
        return Path(globals()["OUT_DIR"])
    if "OUTPUT_DIR" in globals():
        return Path(globals()["OUTPUT_DIR"])
    raise RuntimeError("No existing output-directory variable was found; expected OUT_DIR or OUTPUT_DIR.")


_report_dir = _report_output_dir() / "report"
_report_dir.mkdir(parents=True, exist_ok=True)


def _report_is_inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except Exception:
        return False


def _report_file_sha256(path, allow_heavy=False):
    try:
        path = Path(path)
    except Exception:
        return None
    if not path.exists() or not path.is_file():
        return None
    if not allow_heavy and path.suffix.lower() in {".parquet", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}:
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _report_notebook_path():
    candidates = []
    for key in ["__vsc_ipynb_file__", "__file__"]:
        value = globals().get(key)
        if value:
            candidates.append(Path(value))
    notebook_name = globals().get("NOTEBOOK_NAME")
    if notebook_name:
        candidates.append(Path.cwd() / str(notebook_name))
        project_root = globals().get("PROJECT_ROOT")
        if project_root:
            candidates.append(Path(project_root) / str(notebook_name))
    for candidate in candidates:
        try:
            if candidate.exists() and candidate.suffix.lower() == ".ipynb":
                return candidate
        except Exception:
            pass
    return None


_report_nb_path = _report_notebook_path()
_report_notebook = _report_nb_path.name if _report_nb_path is not None else str(globals().get("NOTEBOOK_NAME", "unknown_notebook"))
_report_category = globals().get("CATEGORY_ID", globals().get("CATEGORY_LABEL", "cross_category"))


def _report_path_from_maps(key):
    for map_name in ["OUTPUT_FILES", "OUTPUT_PATHS", "output_paths"]:
        mapping = globals().get(map_name)
        if isinstance(mapping, dict) and key in mapping:
            return str(mapping[key])
    return None


def _report_path_from_var(name):
    value = globals().get(name)
    return str(value) if value is not None else None


def _report_row_value(row, candidates):
    for column in candidates:
        if column in row and not _report_is_missing(row[column]):
            return row[column]
    return None


def _report_ci(row, low_candidates, high_candidates):
    lo = _report_row_value(row, low_candidates)
    hi = _report_row_value(row, high_candidates)
    if _report_is_missing(lo) or _report_is_missing(hi):
        return None
    return [_report_jsonable(lo), _report_jsonable(hi)]


def _report_value(claim_id, value=None, ci=None, p=None, n=None, source_file=None, aggregation=None, note=None):
    record = {
        "claim_id": str(claim_id),
        "value": _report_jsonable(value),
        "ci": _report_jsonable(ci),
        "p": _report_jsonable(p),
        "n": _report_jsonable(n),
        "source_file": _report_jsonable(source_file),
        "aggregation": _report_jsonable(aggregation),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_missing_value(claim_id, note="not_available_in_notebook"):
    return _report_value(
        claim_id=claim_id,
        value=None,
        ci=None,
        p=None,
        n=None,
        source_file=None,
        aggregation="not_available_in_notebook",
        note=note,
    )


def _report_gate(gate_id, observed=None, expected_contract="", self_flag=False, note=None):
    record = {
        "gate_id": str(gate_id),
        "observed": _report_jsonable(observed),
        "expected_contract": str(expected_contract),
        "self_flag": bool(self_flag),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_frame_failed(frame, passed_columns=("passed", "check_passed", "identity_passed"), status_columns=("status", "coverage_status")):
    if not _report_is_dataframe(frame) or frame.empty:
        return False
    for column in passed_columns:
        if column in frame.columns:
            try:
                return not bool(frame[column].astype(bool).all())
            except Exception:
                pass
    for column in status_columns:
        if column in frame.columns:
            statuses = frame[column].astype(str).str.upper()
            return not bool(statuses.isin(["PASS", "SUCCESS", "TRUE"]).all())
    return False


def _report_filter_primary(frame):
    work = frame.copy()
    original = work
    if "metric_name" in work.columns:
        filtered = work.loc[work["metric_name"].astype(str).eq(str(globals().get("PRIMARY_METRIC_NAME", "NDCG")))]
        if not filtered.empty:
            work = filtered
    if "metric_cutoff" in work.columns:
        filtered = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(int(globals().get("PRIMARY_METRIC_CUTOFF", 5)))]
        if not filtered.empty:
            work = filtered
    if "candidate_pool_depth" in work.columns and "REPORT_POOL_DEPTH" in globals():
        filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["REPORT_POOL_DEPTH"]))]
        if not filtered.empty:
            work = filtered
    return work if not work.empty else original


def _report_values_15():
    values = []
    five = globals().get("stage_allocation_five_condition_comparison_df")
    if _report_is_dataframe(five):
        work = _report_filter_primary(five)
        value_columns = ["mean_metric_value", "metric_mean", "mean_value", "mean", "metric_value_mean", "mean_ndcg_at_5"]
        value_column = next((column for column in value_columns if column in work.columns), None)
        if value_column is None:
            values.append(_report_missing_value("section6_five_condition_means", "value_column_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [
                    row.get("stage_condition"), row.get("reranker_family"),
                    row.get("candidate_pool_depth"), row.get("metric_name"), row.get("metric_cutoff"),
                ]
                claim_id = "section6_condition_mean:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get(value_column),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low", "lower_ci"], ["bootstrap_ci_95_high", "ci_high", "upper_ci"]),
                    p=_report_row_value(row, ["p", "p_value", "sign_flip_p_value"]),
                    n=_report_row_value(row, ["case_count", "query_count", "n_cases", "n", "sample_size"]),
                    source_file=_report_path_from_maps("five_condition_comparison"),
                    aggregation=f"{value_column} from stage_allocation_five_condition_comparison_df",
                ))
    else:
        values.append(_report_missing_value("section6_five_condition_means"))

    contrasts = globals().get("stage_allocation_paired_contrasts_df")
    if _report_is_dataframe(contrasts):
        work = contrasts.copy()
        if "contrast_name" in work.columns:
            work = work.loc[work["contrast_name"].astype(str).eq("P2-P_minus_P2-Q")]
        if "metric_name" in work.columns:
            work = work.loc[work["metric_name"].astype(str).eq("NDCG")]
        if "metric_cutoff" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(5)]
        if "candidate_pool_depth" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").isin([100, 300, 500, 700, 1000])]
        if "user_scope" in work.columns:
            work = work.loc[work["user_scope"].astype(str).eq("overall")]
        if "analysis_subset" in work.columns:
            work = work.loc[work["analysis_subset"].astype(str).eq("all_cases")]
        if work.empty:
            values.append(_report_missing_value("fig6_3_rankp_minus_base_ndcg5_by_depth", "requested_contrast_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                claim_id = "fig6_3_rankp_minus_base_ndcg5_by_depth:" + "|".join(str(row.get(column)) for column in ["reranker_family", "candidate_pool_depth"] if column in row)
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("mean_difference"),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low"], ["bootstrap_ci_95_high", "ci_high"]),
                    p=_report_row_value(row, ["sign_flip_p_value", "p_value", "p"]),
                    n=_report_row_value(row, ["paired_query_count", "n_pairs", "case_count", "n"]),
                    source_file=_report_path_from_maps("paired_contrasts"),
                    aggregation="existing paired mean_difference for P2-P_minus_P2-Q by depth",
                ))
    else:
        values.append(_report_missing_value("fig6_3_rankp_minus_base_ndcg5_by_depth"))
    return values


def _report_gates_15():
    gates = []
    manifest_obj = globals().get("manifest", {}) if isinstance(globals().get("manifest", {}), dict) else {}
    sig = globals().get("stage_allocation_significance_tests_df")
    observed_scope = {
        "analysis_role": manifest_obj.get("analysis_role"),
        "confirmatory_inference_authority_values": sorted(sig["confirmatory_inference_authority"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "confirmatory_inference_authority" in sig.columns else None,
        "inference_role_values": sorted(sig["inference_role"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "inference_role" in sig.columns else None,
    }
    gates.append(_report_gate("descriptive_only_scope", observed_scope, "descriptive-only; confirmatory inference authority remains Notebook 16", False))
    authority = globals().get("aggregate_metric_authority_qc_df")
    gates.append(_report_gate(
        "aggregate_metric_authority_qc",
        _report_frame_records(authority, limit=30),
        "descriptive aggregates reconcile with canonical authority",
        _report_frame_failed(authority, passed_columns=("check_passed", "passed")),
        None if _report_is_dataframe(authority) else "not_available_in_notebook",
    ))
    identity = globals().get("cold_fallback_identity_qc_df")
    gates.append(_report_gate(
        "cold_fallback_identity_qc",
        _report_frame_records(identity, limit=30),
        "fallback identity diagnostics are descriptive QC only",
        _report_frame_failed(identity),
        None if _report_is_dataframe(identity) else "not_available_in_notebook",
    ))
    gates.append(_report_gate(
        "stage_delta_fold_lineage_qc",
        manifest_obj.get("stage_delta_fold_lineage_qc", globals().get("stage_delta_fold_lineage_qc")),
        "stage-delta lineage recorded from existing Notebook 14/15 artifacts",
        False,
        None if (manifest_obj.get("stage_delta_fold_lineage_qc") is not None or "stage_delta_fold_lineage_qc" in globals()) else "not_available_in_notebook",
    ))
    return gates


def _report_values_18():
    values = []
    frame = globals().get("primary_feature_group_gain_summary_df")
    source_key = "primary_feature_group_gain_summary"
    if not _report_is_dataframe(frame):
        frame = globals().get("feature_group_gain_summary_df")
        source_key = "feature_group_gain_summary"
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "primary_report_depth" in work.columns:
            filtered = work.loc[work["primary_report_depth"].astype(bool)]
            if not filtered.empty:
                work = filtered
        elif "candidate_pool_depth" in work.columns and "PRIMARY_POOL_DEPTH" in globals():
            filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["PRIMARY_POOL_DEPTH"]))]
            if not filtered.empty:
                work = filtered
        if "feature_group" not in work.columns or "mean_normalized_gain" not in work.columns:
            values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct", "required_columns_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"fig7_1_grouped_normalized_gain_pct:{row.get('feature_group')}",
                    value=None if _report_is_missing(row.get("mean_normalized_gain")) else float(row.get("mean_normalized_gain")) * 100.0,
                    ci=None,
                    p=None,
                    n=_report_row_value(row, ["fold_count", "n_folds", "n"]),
                    source_file=_report_path_from_maps(source_key),
                    aggregation="mean_normalized_gain across folds, expressed as percent",
                ))
    else:
        values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct"))
    return values


def _report_gates_18():
    gates = []
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    share_observed = {
        "manifest_flag": run.get("grouped_gain_shares_sum_to_one"),
        "primary_share_total": globals().get("_primary_share_total"),
        "group_share_totals": globals().get("_group_share_totals"),
    }
    share_self_flag = False
    if share_observed["manifest_flag"] is not None:
        share_self_flag = share_observed["manifest_flag"] is not True
    gates.append(_report_gate("grouped_gain_shares_sum_to_one", share_observed, "grouped gain shares sum to one within existing fold/depth summaries", share_self_flag))
    perm_flag = run.get("permutation_importance_computed")
    gates.append(_report_gate("permutation_importance_computed", perm_flag, "false in slim build manifest", perm_flag is not False, None if perm_flag is not None else "not_available_in_notebook"))
    shap_flag = run.get("shap_computed")
    gates.append(_report_gate("shap_computed", shap_flag, "false in slim build manifest", shap_flag is not False, None if shap_flag is not None else "not_available_in_notebook"))
    reproduction = globals().get("reproduction_qc_df")
    gates.append(_report_gate(
        "native_fold_prediction_reproduction",
        _report_frame_records(reproduction, limit=30),
        "native fold predictions reproduce strict OOF exports within recorded tolerance",
        _report_frame_failed(reproduction, passed_columns=("prediction_reproduced", "passed")),
        None if _report_is_dataframe(reproduction) else "not_available_in_notebook",
    ))
    return gates


def _report_values_19():
    values = []
    frame = globals().get("branch_contribution_df")
    required = list(globals().get("REQUIRED_BRANCH_GROUPS", ["candidate_common", "functional_prior", "brand_prior"]))
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "branch_group" in work.columns:
            filtered = work.loc[work["branch_group"].astype(str).isin(required)]
            if not filtered.empty:
                work = filtered
        if "importance_mean" not in work.columns:
            values.append(_report_missing_value("table7_1_branch_masking_delta", "importance_mean_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [row.get("branch_group"), row.get("user_scope"), row.get("regime")]
                claim_id = "table7_1_branch_masking_delta:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("importance_mean"),
                    ci=_report_ci(row, ["importance_ci_95_low", "ci_low"], ["importance_ci_95_high", "ci_high"]),
                    p=None,
                    n=_report_row_value(row, ["case_count", "fold_count", "n"]),
                    source_file=_report_path_from_maps("branch_contribution"),
                    aggregation="mean NDCG@5 decrease from existing branch masking summary",
                ))
    else:
        values.append(_report_missing_value("table7_1_branch_masking_delta"))
    return values


def _report_gates_19():
    gates = []
    manifest_obj = globals().get("analysis_manifest", {}) if isinstance(globals().get("analysis_manifest", {}), dict) else {}
    checkpoint = globals().get("checkpoint_diagnostics_df")
    checkpoint_values = checkpoint["checkpoint_selection_metric"].dropna().astype(str).unique().tolist() if _report_is_dataframe(checkpoint) and "checkpoint_selection_metric" in checkpoint.columns else None
    observed_checkpoint = {
        "manifest_checkpoint_selection_metric": manifest_obj.get("checkpoint_selection_metric"),
        "checkpoint_diagnostics_values": checkpoint_values,
    }
    checkpoint_bad = manifest_obj.get("checkpoint_selection_metric") not in (None, "validation_ndcg_at_5")
    if checkpoint_values is not None:
        checkpoint_bad = checkpoint_bad or any(value != "validation_ndcg_at_5" for value in checkpoint_values)
    gates.append(_report_gate("checkpoint_selection_metric", observed_checkpoint, "validation_ndcg_at_5", checkpoint_bad))
    manifests = globals().get("interpretation_manifests", {}) if isinstance(globals().get("interpretation_manifests", {}), dict) else {}
    _full_report_manifest = manifests.get("Full", {}) if isinstance(manifests.get("Full", {}), dict) else {}
    _full_report_role = str(run_manifests.get("Full", {}).get("shared_all_prior_model_role", "train_and_export"))
    observed_mismatch = {
        "analysis_manifest_disabled_category_mismatch": manifest_obj.get("disabled_category_mismatch"),
        "full_manifest_disabled_category_mismatch": _full_report_manifest.get("disabled_category_mismatch"),
        "full_manifest_role": _full_report_role,
    }
    mismatch_bad = manifest_obj.get("disabled_category_mismatch") not in {None, False}
    if _full_report_role == "load_and_score":
        mismatch_bad = mismatch_bad or _full_report_manifest.get("disabled_category_mismatch") is not False
    gates.append(_report_gate("disabled_category_mismatch", observed_mismatch, "false for analysis and load-and-score Full", mismatch_bad))
    return gates


def _report_values_20():
    values = []
    frame = globals().get("scorecard")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            claim_id = "table3_e1_structural_contrast:" + "|".join(str(row.get(column)) for column in ["claim_dimension", "metric"] if column in row)
            values.append(_report_value(
                claim_id=claim_id,
                value=row.get("value"),
                ci=None,
                p=None,
                n=_report_row_value(row, ["n", "case_count", "item_count", "query_count"]),
                source_file=str(_report_output_dir() / "schema_audit_thesis_evidence_scorecard.csv"),
                aggregation="existing schema-audit scorecard value",
            ))
    else:
        values.append(_report_missing_value("table3_e1_structural_contrast_values"))
    return values


def _report_gates_20():
    frame = globals().get("qc_summary")
    return [_report_gate(
        "schema_audit_qc_summary",
        _report_frame_records(frame, limit=50),
        "schema-audit QC rows reported by notebook",
        _report_frame_failed(frame, status_columns=("status",)),
        None if _report_is_dataframe(frame) else "not_available_in_notebook",
    )]


def _report_values_22():
    values = []
    frame = globals().get("paired_summary")
    requested = {
        "Actual_minus_Shuffled",
        "Actual_minus_QCHSfiltered",
        "Actual_minus_QCHS_filtered",
        "QCHS_minus_Actual",
        "Actual_minus_No_User_Brand",
    }
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "population" in work.columns:
            work = work.loc[work["population"].astype(str).eq("non-cold")]
        if "contrast" in work.columns:
            work = work.loc[work["contrast"].astype(str).isin(requested)]
        if work.empty:
            values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas", "requested_non_cold_contrasts_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"table_a_2_prior_policy_delta:{row.get('contrast')}|non-cold",
                    value=row.get("mean_delta"),
                    ci=_report_ci(row, ["ci_low", "bootstrap_ci_95_low"], ["ci_high", "bootstrap_ci_95_high"]),
                    p=_report_row_value(row, ["p", "p_value"]),
                    n=_report_row_value(row, ["n_cases", "n", "case_count"]),
                    source_file=_report_path_from_var("PAIRED_SUMMARY_PATH"),
                    aggregation="existing non-cold paired_summary mean_delta and CI",
                ))
    else:
        values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas"))
    return values


def _report_gates_22():
    gates = []
    paired = globals().get("paired_summary")
    if _report_is_dataframe(paired) and "population" in paired.columns:
        non_cold = paired.loc[paired["population"].astype(str).eq("non-cold")]
        observed_n = _report_frame_records(non_cold, limit=20, columns=["contrast", "population", "n_cases", "n_users"])
    else:
        observed_n = None
    gates.append(_report_gate("non_cold_n", observed_n, "non-cold n recorded in paired_summary", False, None if observed_n is not None else "not_available_in_notebook"))
    model_reuse = globals().get("model_reuse_decision")
    fold_qc = globals().get("fold_qc")
    reuse_observed = {
        "model_reuse_decision": _report_frame_records(model_reuse, limit=20),
        "fold_qc": _report_frame_records(fold_qc, limit=20),
        "canonical_fold_equivalence": globals().get("canonical_fold_equivalence"),
        "canonical_param_equivalence": globals().get("canonical_param_equivalence"),
        "canonical_preprocessing_equivalence": globals().get("canonical_preprocessing_equivalence"),
    }
    gates.append(_report_gate("nb11_fold_hyperparameter_reuse", reuse_observed, "NB11/P2-Q fold and hyperparameter reuse diagnostics are recorded", _report_frame_failed(model_reuse) or _report_frame_failed(fold_qc)))
    seed = globals().get("RANDOM_SEED")
    gates.append(_report_gate("seed", seed, "42", seed not in (None, 42), None if seed is not None else "not_available_in_notebook"))
    return gates


def _report_values_26():
    values = []
    frame = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            values.append(_report_value(
                claim_id=f"section7_1_strong_minus_weak:{row.get('reranker_family')}|{row.get('comparison_id')}",
                value=row.get("strong_minus_weak_delta"),
                ci=_report_ci(row, ["bootstrap_ci_95_lower", "ci_low"], ["bootstrap_ci_95_upper", "ci_high"]),
                p=None,
                n={"strong_case_count": row.get("strong_case_count"), "weak_case_count": row.get("weak_case_count")},
                source_file=_report_path_from_maps("strong_weak_difference_in_delta"),
                aggregation="existing independent strong-minus-weak bootstrap summary",
            ))
    else:
        values.append(_report_missing_value("section7_1_strong_minus_weak_delta_ci"))
    return values


def _report_gates_26():
    gates = []
    coverage = globals().get("regime_pair_coverage_qc")
    gates.append(_report_gate(
        "regime_pair_coverage",
        _report_frame_records(coverage, limit=80),
        "coverage_status PASS for each method x contrast x regime row",
        _report_frame_failed(coverage, status_columns=("coverage_status",)),
        None if _report_is_dataframe(coverage) else "not_available_in_notebook",
    ))
    strong_weak = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(strong_weak):
        observed_overlap = _report_frame_records(strong_weak, limit=30, columns=["reranker_family", "comparison_id", "strong_weak_case_overlap_count", "strong_weak_user_overlap_count"])
        self_flag = False
        for column in ["strong_weak_case_overlap_count", "strong_weak_user_overlap_count"]:
            if column in strong_weak.columns and _report_pd.to_numeric(strong_weak[column], errors="coerce").ne(0).any():
                self_flag = True
    else:
        observed_overlap = None
        self_flag = False
    gates.append(_report_gate("strong_weak_overlap_counts", observed_overlap, "case and user overlap counts equal 0", self_flag, None if observed_overlap is not None else "not_available_in_notebook"))
    return gates


def _report_values_28():
    values = []
    frame = globals().get("table_6_4")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            key = "|".join(str(row.get(column)) for column in ["category_label", "reranker_family", "stage_condition", "candidate_pool_depth"] if column in row)
            values.append(_report_value(
                claim_id=f"table6_4_online_ms_per_query:{key}",
                value=row.get("online_ms_per_query"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="existing online_ms_per_query from table_6_4",
            ))
            values.append(_report_value(
                claim_id=f"table6_4_hardware:{key}",
                value=row.get("compute_device"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="hardware context from existing runtime table/source manifest",
            ))
    else:
        values.append(_report_missing_value("table6_4_method_depth_ms_per_query_and_hardware"))
    return values


def _report_gates_28():
    gates = []
    runtime_qc = globals().get("runtime_qc")
    four_checks = [
        "query_denominator_present",
        "hardware_fields_present",
        "pipeline_stage1_runtime_complete",
        "offline_components_not_in_online_latency",
    ]
    if _report_is_dataframe(runtime_qc) and "check" in runtime_qc.columns:
        raw_four = runtime_qc.loc[runtime_qc["check"].astype(str).isin(four_checks)].copy()
        if raw_four.empty:
            raw_four = runtime_qc.copy()
        observed_qc = _report_frame_records(raw_four, limit=20)
        self_flag = _report_frame_failed(raw_four)
    else:
        observed_qc = None
        self_flag = False
    gates.append(_report_gate("runtime_validation_qc", observed_qc, "four raw runtime validation QC items recorded", self_flag, None if observed_qc is not None else "not_available_in_notebook"))
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    mode = run.get("mode")
    gates.append(_report_gate("aggregation_only", mode, "aggregation_only_no_retrain", mode not in (None, "aggregation_only_no_retrain"), None if mode is not None else "not_available_in_notebook"))
    separation = {
        "online_definition": run.get("online_definition"),
        "offline_excluded": run.get("offline_excluded"),
        "offline_gate": next((row for row in (observed_qc or []) if row.get("check") == "offline_components_not_in_online_latency"), None),
    }
    gates.append(_report_gate("online_offline_separation", separation, "online latency excludes offline components", False if separation["offline_gate"] is not None else True, None if separation["offline_gate"] is not None else "not_available_in_notebook"))
    return gates


def _report_kind():
    name = _report_notebook.lower()
    if name.startswith("15_") or "stage_allocation_summary" in name:
        return "15"
    if name.startswith("18_") or "lightgbm_heldout_interpretation" in name:
        return "18"
    if name.startswith("19_") or "transformer_heldout_interpretation" in name:
        return "19"
    if name.startswith("20_") or "category_schema_audit" in name:
        return "20"
    if name.startswith("22_") or "prior_policy_ablation_lightgbm" in name:
        return "22"
    if name.startswith("26_") or "regime_effect_summary" in name:
        return "26"
    if name.startswith("28_") or "runtime_efficiency_summary" in name:
        return "28"
    return "unknown"


_REPORT_KIND = _report_kind()
_REPORT_SERVED = {
    "15": ["section6_condition_means", "fig6_3"],
    "18": ["fig7_1"],
    "19": ["table7_1", "section7_3"],
    "20": ["table3_e1", "appendix3_e"],
    "22": ["section7_2", "table_a_2"],
    "26": ["section7_1"],
    "28": ["table6_4"],
}.get(_REPORT_KIND, [])
_REPORT_VALUE_BUILDERS = {
    "15": _report_values_15,
    "18": _report_values_18,
    "19": _report_values_19,
    "20": _report_values_20,
    "22": _report_values_22,
    "26": _report_values_26,
    "28": _report_values_28,
}
_REPORT_GATE_BUILDERS = {
    "15": _report_gates_15,
    "18": _report_gates_18,
    "19": _report_gates_19,
    "20": _report_gates_20,
    "22": _report_gates_22,
    "26": _report_gates_26,
    "28": _report_gates_28,
}


def _report_collect_inputs():
    records = []
    seen = set()

    def add_path(path, sha=None):
        if _report_is_missing(path):
            return
        text = str(path).strip()
        if not text:
            return
        candidate = Path(text)
        if _report_is_inside(candidate, _report_output_dir()):
            return
        key = str(candidate)
        if key in seen:
            return
        seen.add(key)
        records.append({"path": key, "sha256": _report_jsonable(sha if sha is not None else _report_file_sha256(candidate, allow_heavy=False))})

    def looks_like_path(text):
        suffix = Path(str(text)).suffix.lower()
        return suffix in {".json", ".csv", ".parquet", ".txt", ".yaml", ".yml", ".ipynb", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}

    def walk(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if isinstance(value, (str, Path)) and (key_text.endswith("path") or key_text.endswith("paths") or looks_like_path(value)):
                    sha = None
                    if key_text.endswith("path"):
                        sha = obj.get(str(key).replace("path", "sha256")) or obj.get(str(key).replace("_path", "_sha256"))
                    add_path(value, sha)
                else:
                    walk(value)
        elif isinstance(obj, (list, tuple, set)):
            for value in obj:
                walk(value)
        elif isinstance(obj, (str, Path)) and looks_like_path(obj):
            add_path(obj)

    for obj_name in ["pipeline_manifest", "run_manifest", "analysis_manifest", "manifest"]:
        obj = globals().get(obj_name)
        if isinstance(obj, dict):
            walk(obj)

    inventory = globals().get("runtime_inventory")
    if _report_is_dataframe(inventory):
        for _, row in inventory.iterrows():
            for path_col in ["source_path", "source_manifest", "manifest_path", "path"]:
                if path_col in inventory.columns:
                    sha = None
                    for sha_col in [path_col.replace("path", "sha256"), "source_sha256", "sha256"]:
                        if sha_col in inventory.columns:
                            sha = row.get(sha_col)
                            break
                    add_path(row.get(path_col), sha)

    for name, value in list(globals().items()):
        if not name.endswith("_PATH"):
            continue
        if name.startswith(("OUTPUT", "PREDICTIONS", "PER_CASE", "DELTAS", "PAIRED_SUMMAR", "POPULATION_RESULTS", "PROFILE_DIAGNOSTICS", "SHUFFLE_ASSIGNMENT", "FALLBACK_QC", "FEATURE_CONTRACT", "MODEL_REUSE", "FOLD_QC", "LEAKAGE_QC", "COVERAGE", "RETENTION", "FEATURE_IMPORTANCE", "QC_SUMMAR", "MANIFEST")):
            if _report_is_inside(value, _report_output_dir()):
                continue
        add_path(value)

    return records


_report_run_utc = datetime.now(timezone.utc).isoformat()
_report_values = _REPORT_VALUE_BUILDERS.get(_REPORT_KIND, lambda: [_report_missing_value("requested_values")])()
_report_gates = _REPORT_GATE_BUILDERS.get(_REPORT_KIND, lambda: [_report_gate("requested_gates", None, "not_available_in_notebook", False, "not_available_in_notebook")])()
_report_lineage = {
    "notebook": _report_notebook,
    "category": _report_jsonable(_report_category),
    "run_utc": _report_run_utc,
    "code_sha": _report_file_sha256(_report_nb_path, allow_heavy=True) if _report_nb_path is not None else None,
    "inputs": _report_collect_inputs(),
}


def _report_md_cell(value):
    text = "" if value is None else (_report_jsonable(value))
    if isinstance(text, (dict, list)):
        text = json.dumps(text, ensure_ascii=False, sort_keys=True)
    text = str(text)
    return text.replace("|", "\\|").replace("\n", "<br>")


def _report_markdown(values, gates, lineage):
    lines = []
    lines.append("# Identity & lineage")
    lines.append(f"- notebook: {_report_md_cell(lineage.get('notebook'))}")
    lines.append(f"- category: {_report_md_cell(lineage.get('category'))}")
    lines.append(f"- run_utc: {_report_md_cell(lineage.get('run_utc'))}")
    lines.append(f"- code_sha: {_report_md_cell(lineage.get('code_sha'))}")
    lines.append(f"- input_count: {len(lineage.get('inputs', []))}")
    lines.append("")
    lines.append("# Served thesis elements")
    if _REPORT_SERVED:
        for served_id in _REPORT_SERVED:
            lines.append(f"- {_report_md_cell(served_id)}")
    else:
        lines.append("- not_available_in_notebook")
    lines.append("")
    lines.append("# Computed headline values")
    value_columns = ["claim_id", "value", "ci", "p", "n", "source_file", "aggregation", "thesis_value"]
    lines.append("| " + " | ".join(value_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(value_columns)) + " |")
    for record in values:
        row = dict(record)
        row["thesis_value"] = ""
        lines.append("| " + " | ".join(_report_md_cell(row.get(column)) for column in value_columns) + " |")
    lines.append("")
    lines.append("# QC gates")
    gate_columns = ["gate_id", "observed", "expected_contract", "self_flag"]
    lines.append("| " + " | ".join(gate_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(gate_columns)) + " |")
    for record in gates:
        lines.append("| " + " | ".join(_report_md_cell(record.get(column)) for column in gate_columns) + " |")
    lines.append("")
    lines.append("# Self-detected anomalies")
    anomalies = []
    for record in gates:
        if record.get("self_flag") is True:
            anomalies.append(f"- gate_self_flag: {_report_md_cell(record.get('gate_id'))}")
        if record.get("note"):
            anomalies.append(f"- gate_note: {_report_md_cell(record.get('gate_id'))}: {_report_md_cell(record.get('note'))}")
    for record in values:
        if record.get("note"):
            anomalies.append(f"- value_note: {_report_md_cell(record.get('claim_id'))}: {_report_md_cell(record.get('note'))}")
    lines.extend(anomalies if anomalies else ["- none"])
    lines.append("")
    return "\n".join(lines)


_lineage_path = _report_dir / "lineage.json"
_values_path = _report_dir / "report_values.json"
_gates_path = _report_dir / "qc_gates.json"
_markdown_path = _report_dir / "verification_report.md"
_lineage_path.write_text(json.dumps(_report_lineage, ensure_ascii=False, indent=2), encoding="utf-8")
_values_path.write_text(json.dumps(_report_values, ensure_ascii=False, indent=2), encoding="utf-8")
_gates_path.write_text(json.dumps(_report_gates, ensure_ascii=False, indent=2), encoding="utf-8")
_markdown_path.write_text(_report_markdown(_report_values, _report_gates, _report_lineage), encoding="utf-8")
for _written_path in [_lineage_path, _values_path, _gates_path, _markdown_path]:
    print(_written_path)


/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/report/lineage.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/report/report_values.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/report/qc_gates.json
/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/transformer_interpretation/report/verification_report.md
